<a href="https://colab.research.google.com/github/yi-juho/AI_LLM_BERT/blob/main/KAN_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install git+https://github.com/KindXiaoming/pykan.git

In [ ]:
pip install pykan

In [ ]:
# requirements.txt 파일 생성
requirements = """\
matplotlib==3.6.2
numpy==1.24.4
scikit_learn==1.1.3
setuptools==65.5.0
sympy==1.11.1
torch==2.0.1
torchvision==0.15.2
tqdm==4.66.2
pandas==2.0.1
seaborn
pyyaml
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

# 패키지 설치
!pip install -r requirements.txt


#### KAN test

In [ ]:
from kan import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# create a KAN: 2D inputs, 1D output, and 5 hidden neurons. cubic spline (k=3), 5 grid intervals (grid=5).
model = KAN(width=[4,2,1,1], grid=3, k=3, seed=1, device=device)
f = lambda x: torch.exp((torch.sin(torch.pi*(x[:,[0]]**2+x[:,[1]]**2))+torch.sin(torch.pi*(x[:,[2]]**2+x[:,[3]]**2)))/2)
dataset = create_dataset(f, n_var=4, train_num=3000, device=device)

# train the model
model.fit(dataset, opt="LBFGS", steps=20, lamb=0.002, lamb_entropy=2.);

In [ ]:
from kan import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# create a KAN: 2D inputs, 1D output, and 5 hidden neurons. cubic spline (k=3), 5 grid intervals (grid=5).
model = KAN(width=[4,2,1,1], grid=5, k=5, seed=1, device=device)
f = lambda x: torch.exp((torch.sin(torch.pi*(x[:,[0]]**2+x[:,[1]]**2))+torch.sin(torch.pi*(x[:,[2]]**2+x[:,[3]]**2)))/2)
dataset = create_dataset(f, n_var=4, train_num=3000, device=device)

# train the model
model.fit(dataset, opt="LBFGS", steps=200, lamb=0.002, lamb_entropy=2.);

In [ ]:
from kan import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# create a KAN: 2D inputs, 1D output, and 5 hidden neurons. cubic spline (k=3), 5 grid intervals (grid=5).
model = KAN(width=[1,3,1], grid=3, k=3, seed=1, device=device)
f = lambda x: x**3
dataset = create_dataset(f, n_var=1, train_num=3000, device=device)

# train the model
model.fit(dataset, opt="LBFGS", steps=20, lamb=0.002, lamb_entropy=2.);

In [ ]:
x_new = torch.tensor([
    [-1.0],
    [-0.5],
    [0],
    [0.5],
    [0.8]
], dtype=torch.float32).to(device)


In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(x_new)

print("추론 결과:", y_pred)

In [ ]:
model = model.prune(edge_th=1e-2)

In [ ]:
model.plot()

In [ ]:
model.auto_symbolic()

In [ ]:
# input (5 sample, 4dim)
x_new = torch.tensor([
    [0.5, 0.3, -0.1, 0.2],
    [1.0, 0.0, 0.0, 1.0],
    [-0.7, 0.4, 0.2, -0.3],
    [2.0, 1.0, 0.5, 1.2],
    [-1.5, 0.2, 0.3, 0.9]
], dtype=torch.float32).to(device)

model.eval()  # (옵션) 학습 시 dropout/BN 등을 사용했다면 eval()로 설정
with torch.no_grad():
    y_pred = model(x_new)
print("추론 결과:", y_pred)

#### 구조

##### spline

In [ ]:
#print(f"[DEBUG] bit‑spline 호출: x_bit={x_bit[:8]}…, step={step}, order={order}")
# 이거 def b_spline_basis_bits(x_bit: str, grid: list, step: float, order: int) -> list:
#바로 아래에 추가하면 바뀌었는지 호출시키는 코드

In [ ]:
# check_structure.py
import os, sys, importlib, inspect

# 1) 프로젝트 루트 경로에 내 모듈이 있는 디렉토리 추가
sys.path.append(os.getcwd())

# 2) 내 bit‑spline 모듈 불러오기(리로드)
import my_bit_spline
importlib.reload(my_bit_spline)

# 3) KANLinear 클래스와 b_splines 메서드 가져오기
from my_bit_spline import KANLinear

# 4) 클래스가 정의된 파일 경로 확인
print("KANLinear 클래스 정의 파일:",
      inspect.getsourcefile(KANLinear))

# 5) b_splines 메서드가 정의된 파일과 라인 확인
fn = KANLinear.b_splines
print("b_splines 메서드 정의 파일:",
      inspect.getsourcefile(fn))
print("b_splines 시작 라인:", fn.__code__.co_firstlineno)

# 6) b_splines 앞부분 소스 보기
print("\n— b_splines 소스 (처음 20줄) —")
src = inspect.getsource(fn).splitlines()
for line in src[:20]:
    print(line)


In [ ]:
import inspect
import torch
from my_bit_spline import KANLinear  # 방금 만든 파일 이름에 맞춰 바꿔주세요

# 1) 소스코드 확인
print("— KANLinear.b_splines source —")
print(inspect.getsource(KANLinear.b_splines))

# 2) shape 검증
lin = KANLinear(in_features=2, out_features=4, grid_size=5, spline_order=3)
x = torch.linspace(-1, 1, steps=5).unsqueeze(-1).repeat(1, 2)  # (5,2)
B = lin.b_splines(x)

expected = (x.shape[0], lin.in_features, lin.grid_size + lin.spline_order)  # (5,2,8)
print("\n— b_splines output shape —")
print("expected:", expected)
print("actual:  ", tuple(B.shape))

assert B.shape == expected, "❌ b_splines 출력 shape가 예상과 다릅니다!"
print("✅ b_splines 가 bit-based 구현으로 잘 교체되었습니다.")



In [ ]:
import os, sys
sys.path.append(os.getcwd())


In [ ]:
from my_bit_spline import KAN


In [ ]:
import importlib, my_bit_spline
importlib.reload(my_bit_spline)
from my_bit_spline import KAN


In [ ]:
%%bash
cat > my_bit_spline.py << 'EOF'
import math
import torch
import torch.nn.functional as F

# ——————————————————————————————————————
# 1) 32비트 고정소수점 변환/해석
# ——————————————————————————————————————
def float_to_custom_bits(value: float) -> str:
    """실수 → 1:11:20 고정소수점 32비트 2의 보수 비트열."""
    scale = 1 << 20
    ival = int(round(value * scale))
    return format(ival & 0xFFFFFFFF, '032b')

def custom_bits_to_float(b: str) -> float:
    """32비트 2의 보수 고정소수점 → 실수."""
    if len(b) != 32:
        raise ValueError("32비트 문자열 필요")
    ival = int(b, 2)
    if b[0] == '1':
        ival -= 1 << 32
    return ival / (1 << 20)

def bits_to_int(b: str) -> int:
    ival = int(b, 2)
    return ival - (1 << 32) if b[0] == '1' else ival

def int_to_bits(i: int) -> str:
    return format(i & 0xFFFFFFFF, '032b')

# ——————————————————————————————————————
# 2) 비트 단위 사칙연산
# ——————————————————————————————————————
def add_bits(a: str, b: str) -> str:
    return int_to_bits(bits_to_int(a) + bits_to_int(b))

def sub_bits(a: str, b: str) -> str:
    return int_to_bits(bits_to_int(a) - bits_to_int(b))

def mul_bits(a: str, b: str) -> str:
    prod = (bits_to_int(a) * bits_to_int(b)) >> 20
    return int_to_bits(prod)

def div_bits(a: str, b: str) -> str:
    num = bits_to_int(a) << 20
    den = bits_to_int(b)
    quot = num // den if den != 0 else 0
    return int_to_bits(quot)

# ——————————————————————————————————————
# 3) Grid 생성 (knot 확장)
# ——————————————————————————————————————
def make_spline_grid(n_intervals: int, order: int, lo=-1.0, hi=1.0):
    step = (hi - lo) / n_intervals
    base = [lo + i * step for i in range(n_intervals + 1)]
    left  = [lo - step * (order - i) for i in range(order)]
    right = [hi + step * (i + 1)       for i in range(order)]
    full  = left + base + right
    return full, step

# ——————————————————————————————————————
# 4) B‑스플라인 기저함수 계산 (LUT 기반 reciprocal)
# ——————————————————————————————————————
def b_spline_basis_bits(x_bit: str, grid: list, step: float, order: int) -> list:
    # 디버깅
    # print(f"[DEBUG] bit‑spline 호출: x_bit={x_bit[:8]}…, step={step}, order={order}")
    grid_bits = [ float_to_custom_bits(v) for v in grid ]
    N = len(grid_bits) - 1

    recips = []
    for p in range(1, order+1):
        recips.append(
          float_to_custom_bits(1.0 / (p * step)) if p*step!=0 else float_to_custom_bits(0.0)
        )

    B = []
    xi = bits_to_int(x_bit)
    for i in range(N):
        lo_i = bits_to_int(grid_bits[i])
        hi_i = bits_to_int(grid_bits[i+1])
        B.append(float_to_custom_bits(1.0) if lo_i <= xi < hi_i else float_to_custom_bits(0.0))

    for p in range(1, order+1):
        recip = recips[p-1]
        Bp = []
        for i in range(len(B)-1):
            t_i    = grid_bits[i]
            t_ip1  = grid_bits[i+1]
            t_i_p  = grid_bits[i+p]
            t_ip1_p= grid_bits[i+p+1]

            num1   = sub_bits(x_bit, t_i)
            frac1  = mul_bits(num1, recip)
            term1  = mul_bits(frac1, B[i])

            num2   = sub_bits(t_ip1_p, x_bit)
            frac2  = mul_bits(num2, recip)
            term2  = mul_bits(frac2, B[i+1])

            Bp.append(add_bits(term1, term2))
        B = Bp
    return B

# ——————————————————————————————————————
# 5) KANLinear / KAN 클래스 (bit‑based spline)
# ——————————————————————————————————————
class KANLinear(torch.nn.Module):
    def __init__(self, in_features, out_features,
                 grid_size=5, spline_order=3,
                 scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
                 enable_standalone_scale_spline=True,
                 base_activation=torch.nn.Identity,
                 grid_eps=0.02, grid_range=[-1,1]):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.grid_size    = grid_size
        self.spline_order = spline_order

        h = (grid_range[1]-grid_range[0]) / grid_size
        grid = (
          (torch.arange(-spline_order, grid_size+spline_order+1)*h + grid_range[0])
          .expand(in_features,-1).contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight   = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size+spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise  = scale_noise
        self.scale_base   = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps     = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5)*self.scale_base)
        with torch.no_grad():
            noise = ((torch.rand(self.grid_size+1, self.in_features, self.out_features)-0.5)
                     * self.scale_noise/self.grid_size)
            coeffs = self.curve2coeff(self.grid.T[self.spline_order:-self.spline_order], noise)
            if self.enable_standalone_scale_spline:
                coeffs *= self.scale_spline
            self.spline_weight.data.copy_(coeffs)
            if self.enable_standalone_scale_spline:
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5)*self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        # torch-based 원본 spline
        assert x.dim()==2 and x.size(1)==self.in_features
        out = []
        for batch_row in x:
            row_bs = []
            for i in range(batch_row.size(0)):
                bit = float_to_custom_bits(batch_row[i].item())
                row_bs.append(
                    b_spline_basis_bits(bit,
                       self.grid[i].tolist(),
                       (self.grid[0,1]-self.grid[0,0]).item(),
                       self.spline_order
                    )
                )
            # row_bs: list of length in_features, each is [bits...]
            # stack → tensor (in, coeff)
            rows = torch.tensor(
                [[custom_bits_to_float(b) for b in feat] for feat in row_bs],
                dtype=x.dtype, device=x.device
            )
            out.append(rows)
        # out: list of batch each (in, coeff) → (batch, in, coeff)
        return torch.stack(out, dim=0)

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        A = self.b_splines(x).transpose(0,1)
        B = y.transpose(0,1)
        sol = torch.linalg.lstsq(A,B).solution
        return sol.permute(2,0,1).contiguous()

    @property
    def scaled_spline_weight(self):
        if self.enable_standalone_scale_spline:
            return self.spline_weight * self.spline_scaler.unsqueeze(-1)
        else:
            return self.spline_weight

    def forward(self, x: torch.Tensor):
        assert x.size(-1)==self.in_features
        orig = x.shape
        x = x.reshape(-1,self.in_features)

        base_out = F.linear(self.base_activation(x), self.base_weight)
        sp_out   = F.linear(
            self.b_splines(x).view(x.size(0),-1),
            self.scaled_spline_weight.view(self.out_features,-1)
        )
        out = base_out + sp_out
        return out.reshape(*orig[:-1], self.out_features)

class KAN(torch.nn.Module):
    def __init__(self, layers_hidden, **kw):
        super().__init__()
        self.layers = torch.nn.ModuleList([
            KANLinear(in_f,out_f,**kw)
            for in_f,out_f in zip(layers_hidden, layers_hidden[1:])
        ])
    def forward(self, x, update_grid=False):
        for L in self.layers:
            if update_grid: L.update_grid(x)
            x = L(x)
        return x
EOF


In [ ]:
%%bash
cat > my_bit_spline.py << 'EOF'
import math
import torch
import torch.nn.functional as F

# ——————————————————————————————
# Q5.6: INT_BITS=5(부호 포함), FRAC_BITS=6 → TOTAL_BITS=11
# 범위: [-16, 15 + 63/64] = [-16.0, 15.984375]
# ——————————————————————————————
INT_BITS  = 5
FRAC_BITS = 6
TOTAL_BITS = INT_BITS + FRAC_BITS
MASK = (1 << TOTAL_BITS) - 1

def float_to_custom_bits(value: float) -> str:
    scale = 1 << FRAC_BITS
    ival = int(round(value * scale))
    return format(ival & MASK, f'0{TOTAL_BITS}b')

def custom_bits_to_float(b: str) -> float:
    if len(b) != TOTAL_BITS:
        raise ValueError(f"{TOTAL_BITS}비트 문자열 필요")
    ival = int(b, 2)
    if b[0] == '1':
        ival -= 1 << TOTAL_BITS
    return ival / (1 << FRAC_BITS)

def bits_to_int(b: str) -> int:
    if len(b) != TOTAL_BITS:
        raise ValueError(f"{TOTAL_BITS}비트 문자열 필요")
    ival = int(b, 2)
    return ival - (1 << TOTAL_BITS) if b[0] == '1' else ival

def int_to_bits(i: int) -> str:
    return format(i & MASK, f'0{TOTAL_BITS}b')

def add_bits(a: str, b: str) -> str:
    return int_to_bits(bits_to_int(a) + bits_to_int(b))

def sub_bits(a: str, b: str) -> str:
    return int_to_bits(bits_to_int(a) - bits_to_int(b))

def mul_bits(a: str, b: str) -> str:
    prod = (bits_to_int(a) * bits_to_int(b)) >> FRAC_BITS
    return int_to_bits(prod)

def div_bits(a: str, b: str) -> str:
    num = bits_to_int(a) << FRAC_BITS
    den = bits_to_int(b)
    quot = num // den if den != 0 else 0
    return int_to_bits(quot)

def make_spline_grid(n_intervals: int, order: int, lo=-1.0, hi=1.0):
    step = (hi - lo) / n_intervals
    base = [lo + i * step for i in range(n_intervals + 1)]
    left  = [lo - step * (order - i) for i in range(order)]
    right = [hi + step * (i + 1) for i in range(order)]
    full  = left + base + right
    return full, step

def b_spline_basis_bits(x_bit: str, grid: list, step: float, order: int) -> list:
    grid_bits = [ float_to_custom_bits(v) for v in grid ]
    N = len(grid_bits) - 1
    recips = [ float_to_custom_bits(1.0/(p*step)) if p*step!=0 else float_to_custom_bits(0.0)
               for p in range(1, order+1) ]

    B = []
    xi = bits_to_int(x_bit)
    for i in range(N):
        lo_i = bits_to_int(grid_bits[i])
        hi_i = bits_to_int(grid_bits[i+1])
        B.append(float_to_custom_bits(1.0) if lo_i <= xi < hi_i else float_to_custom_bits(0.0))

    for p in range(1, order+1):
        recip = recips[p-1]
        Bp = []
        for i in range(len(B)-1):
            t_i, t_ip1 = grid_bits[i], grid_bits[i+1]
            t_i_p, t_ip1_p = grid_bits[i+p], grid_bits[i+p+1]

            term1 = mul_bits(mul_bits(sub_bits(x_bit, t_i), recip), B[i])
            term2 = mul_bits(mul_bits(sub_bits(t_ip1_p, x_bit), recip), B[i+1])
            Bp.append(add_bits(term1, term2))
        B = Bp
    return B

class KANLinear(torch.nn.Module):
    def __init__(self, in_features, out_features,
                 grid_size=5, spline_order=3,
                 scale_noise=0.1, scale_base=1.0, scale_spline=1.0,
                 enable_standalone_scale_spline=True,
                 base_activation=torch.nn.Identity,
                 grid_eps=0.02, grid_range=[-1,1]):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.grid_size    = grid_size
        self.spline_order = spline_order

        h = (grid_range[1]-grid_range[0]) / grid_size
        grid = ((torch.arange(-spline_order, grid_size+spline_order+1)*h + grid_range[0])
                .expand(in_features,-1).contiguous())
        self.register_buffer("grid", grid)

        self.base_weight   = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features, grid_size+spline_order))
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(torch.Tensor(out_features, in_features))

        self.scale_noise  = scale_noise
        self.scale_base   = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps     = grid_eps
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5)*self.scale_base)
        with torch.no_grad():
            noise = ((torch.rand(self.grid_size+1, self.in_features, self.out_features)-0.5)
                     * self.scale_noise/self.grid_size)
            coeffs = self.curve2coeff(self.grid.T[self.spline_order:-self.spline_order], noise)
            if self.enable_standalone_scale_spline:
                coeffs *= self.scale_spline
            self.spline_weight.data.copy_(coeffs)
            if self.enable_standalone_scale_spline:
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5)*self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        assert x.dim()==2 and x.size(1)==self.in_features
        out = []
        step = (self.grid[0,1]-self.grid[0,0]).item()
        for batch_row in x:
            row_bs = []
            for i in range(batch_row.size(0)):
                bit = float_to_custom_bits(batch_row[i].item())
                row_bs.append(b_spline_basis_bits(bit, self.grid[i].tolist(), step, self.spline_order))
            rows = torch.tensor([[custom_bits_to_float(b) for b in feat] for feat in row_bs],
                                dtype=x.dtype, device=x.device)
            out.append(rows)
        return torch.stack(out, dim=0)

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        A = self.b_splines(x).transpose(0,1)
        B = y.transpose(0,1)
        sol = torch.linalg.lstsq(A,B).solution
        return sol.permute(2,0,1).contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (self.spline_scaler.unsqueeze(-1) if self.enable_standalone_scale_spline else 1.0)

    def forward(self, x: torch.Tensor):
        assert x.size(-1)==self.in_features
        orig = x.shape
        x = x.reshape(-1,self.in_features)
        base_out = F.linear(self.base_activation(x), self.base_weight)
        sp_out   = F.linear(self.b_splines(x).view(x.size(0),-1),
                            self.scaled_spline_weight.view(self.out_features,-1))
        out = base_out + sp_out
        return out.reshape(*orig[:-1], self.out_features)

class KAN(torch.nn.Module):
    def __init__(self, layers_hidden, **kw):
        super().__init__()
        self.layers = torch.nn.ModuleList([KANLinear(in_f,out_f,**kw)
                                           for in_f,out_f in zip(layers_hidden, layers_hidden[1:])])
    def forward(self, x, update_grid=False):
        for L in self.layers:
            if update_grid: L.update_grid(x)
            x = L(x)
        return x
EOF


##### sliu

In [ ]:
import torch
import torch.nn as nn
class PadeSiluApprox(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        num = 1 - 0.5 * x + (x ** 2) / 10 - (x ** 3) / 120
        den = 1 + 0.5 * x + (x ** 2) / 10 + (x ** 3) / 120
        exp_m_x = num / den
        return x / (1 + exp_m_x)

In [ ]:
import torch
import torch.nn as nn
class PadeSiluApprox1(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        numerator = 1 - (0.5 * x) + (x * x) / 12 - (x * x * x) / 24
        denominator = 1 + (0.5 * x) + (x * x) / 12 + (x * x * x) / 24
        exp_m_x = numerator / denominator
        return x / (1 + exp_m_x)

In [ ]:
import torch
import torch.nn.functional as F
import math


class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=PadeSiluApprox,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        x = x.reshape(-1, self.in_features)

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output

        output = output.reshape(*original_shape[:-1], self.out_features)
        return output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin=0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype=torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )


class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        base_activation=PadeSiluApprox,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KAN, self).__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

In [ ]:
# Train on MNIST
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# Load MNIST
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)
trainset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
valset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
valloader = DataLoader(valset, batch_size=64, shuffle=False)

# Define model
model = KAN([28 * 28, 64, 10])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# Define optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
# Define learning rate scheduler
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)

# Define loss
criterion = nn.CrossEntropyLoss()
for epoch in range(10):
    # Train
    model.train()
    with tqdm(trainloader) as pbar:
        for i, (images, labels) in enumerate(pbar):
            images = images.view(-1, 28 * 28).to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels.to(device))
            loss.backward()
            optimizer.step()
            accuracy = (output.argmax(dim=1) == labels.to(device)).float().mean()
            pbar.set_postfix(loss=loss.item(), accuracy=accuracy.item(), lr=optimizer.param_groups[0]['lr'])

    # Validation
    model.eval()
    val_loss = 0
    val_accuracy = 0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.view(-1, 28 * 28).to(device)
            output = model(images)
            val_loss += criterion(output, labels.to(device)).item()
            val_accuracy += (
                (output.argmax(dim=1) == labels.to(device)).float().mean().item()
            )
    val_loss /= len(valloader)
    val_accuracy /= len(valloader)

    # Update learning rate
    scheduler.step()

    print(
        f"Epoch {epoch + 1}, Val Loss: {val_loss}, Val Accuracy: {val_accuracy}"
    )

In [ ]:
import torch
import torch.nn.functional as F
import math


class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        x = x.reshape(-1, self.in_features)

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output

        output = output.reshape(*original_shape[:-1], self.out_features)
        return output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin=0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype=torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )


class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KAN, self).__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

In [ ]:
import torch

# 1. device 정의
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 모델 생성
kan = KAN([784, 64, 10], grid_size=5, spline_order=3).to(device)

# 3. 첫 레이어 in_features 확인
layer0 = kan.layers[0]
print("이 레이어의 in_features =", layer0.in_features)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import math
import torch.nn.functional as F

# Define the KANLinear class
class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        x = x.unsqueeze(-1)
        bases = ((x >= self.grid[:, :-1]) & (x < self.grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - self.grid[:, : -(k + 1)])
                / (self.grid[:, k:-1] - self.grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (self.grid[:, k + 1 :] - x)
                / (self.grid[:, k + 1 :] - self.grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        A = self.b_splines(x).transpose(0, 1)
        B = y.transpose(0, 1)
        solution = torch.linalg.lstsq(A, B).solution
        result = solution.permute(2, 0, 1)
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        x = x.reshape(-1, self.in_features)
        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output
        return output

# Define the KAN class
class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KAN, self).__init__()
        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )
            )

    def forward(self, x: torch.Tensor):
        for layer in self.layers:
            x = layer(x)
        return x

# Load MNIST dataset and train KAN model
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)
trainset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
valset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
valloader = DataLoader(valset, batch_size=64, shuffle=False)

# Define the model, optimizer, scheduler, and loss function
model = KAN([28 * 28, 64, 10])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(10):
    model.train()
    with tqdm(trainloader) as pbar:
        for images, labels in pbar:
            images = images.view(-1, 28 * 28).to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels.to(device))
            loss.backward()
            optimizer.step()
            accuracy = (output.argmax(dim=1) == labels.to(device)).float().mean()
            pbar.set_postfix(loss=loss.item(), accuracy=accuracy.item(), lr=optimizer.param_groups[0]['lr'])

    # Validation loop
    model.eval()
    val_loss = 0
    val_accuracy = 0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.view(-1, 28 * 28).to(device)
            output = model(images)
            val_loss += criterion(output, labels.to(device)).item()
            val_accuracy += (output.argmax(dim=1) == labels.to(device)).float().mean().item()
    val_loss /= len(valloader)
    val_accuracy /= len(valloader)

    # Update learning rate
    scheduler.step()

    print(f"Epoch {epoch + 1}, Val Loss: {val_loss}, Val Accuracy: {val_accuracy}")

#### val

In [ ]:
import importlib, my_bit_spline
my_bit_spline = importlib.reload(my_bit_spline)   # 파일 수정했으면 갱신
BitKAN = my_bit_spline.KAN

bit_model = BitKAN([28*28, 64, 10], base_activation=PadeSiluApprox)

import inspect
print("bit_model class module:", type(bit_model).__module__)
print("first layer class module:", type(bit_model.layers[0]).__module__)
print("b_splines defined in:", bit_model.layers[0].b_splines.__func__.__module__)
print("b_splines file:", inspect.getsourcefile(bit_model.layers[0].b_splines))


In [ ]:
#model = KAN([28 * 28, 64, 10],
#            grid_size=5,
#            spline_order=3,
#            base_activation=torch.nn.SiLU)
print(type(model).__module__)
print(type(model.layers[0]).__module__)

In [ ]:
model = KAN([28 * 28, 64, 10])

In [ ]:
import torch
torch.save(model.state_dict(),"./model/efficient_KAN_mnist.pt")

In [ ]:
model.load_state_dict(torch.load("./model/efficient_KAN_mnist.pt"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# 1) 데이터
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,),(0.5,))])
valset = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)
valloader = DataLoader(valset, batch_size=64, shuffle=False)

# 2) 디바이스 & 모델 생성 → 가중치 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = KAN([28*28, 64, 10])  # 학습 때와 동일!
state = torch.load("mnist_kan_float.pth", map_location="cpu")  # 저장 파일 경로
# DataParallel 저장을 대비해 'module.' 접두어 제거
if any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.","",1): v for k,v in state.items()}
model.load_state_dict(state, strict=True)
model.to(device).eval()

# 3) 검증 (루프에서만 flatten)
criterion = nn.CrossEntropyLoss()
val_loss_sum=0.0; val_correct=0; val_total=0
with torch.inference_mode():
    for images, labels in tqdm(valloader, desc="Validating", unit="batch"):
        images = images.view(-1, 28*28).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        val_loss_sum += loss.item() * images.size(0)
        val_correct  += (logits.argmax(1) == labels).sum().item()
        val_total    += images.size(0)
val_loss = val_loss_sum/val_total
val_acc  = val_correct/val_total
print(f"\n[LOAD&VAL] Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
# =========================
# 학습 직후: 최종 검증 스크립트
# =========================

# 1) 가중치 저장(장치 꼬임 방지 위해 CPU로 저장)
#save_path = "mnist_kan_float.pth"
#torch.save(model.cpu().state_dict(), save_path)
model.to(device).eval()  # 원래 모델도 장치 복귀

# 2) 동일 아키텍처의 새 모델 생성 후 로드
fresh_model = KAN([28 * 28, 64, 10]).to(device)
fresh_model.load_state_dict(torch.load(save_path, map_location="cpu"), strict=True)
fresh_model.eval()

# (선택) 저장 전/후 동치 체크: 같은 배치에 대해 출력 차이 확인
imgs0, _ = next(iter(valloader))
with torch.inference_mode():
    xa = imgs0.view(-1, 28 * 28).to(device)
    ya_old = model(xa)
    ya_new = fresh_model(xa)
    print(f"[Sanity] max |old-new| = {(ya_old - ya_new).abs().max().item():.6e}")

# 3) 최종 Validation
criterion = nn.CrossEntropyLoss()
val_loss_sum = 0.0
val_correct  = 0
val_total    = 0

with torch.inference_mode():
    for images, labels in tqdm(valloader, desc="Final validation", unit="batch"):
        images = images.view(-1, 28 * 28).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = fresh_model(images)
        loss = criterion(logits, labels)

        val_loss_sum += loss.item() * images.size(0)
        val_correct  += (logits.argmax(1) == labels).sum().item()
        val_total    += images.size(0)

val_loss = val_loss_sum / val_total
val_acc  = val_correct / val_total
print(f"\n[FINAL] Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc*100:.2f}%")


In [ ]:
# validate_mnist_kan.py
import torch
import torch.nn as nn
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 0) KAN 불러오기 (네가 KAN을 정의한 파일/모듈로 변경)
# from my_float_kan import KAN
from your_kan_file import KAN   # ← 여기만 네 파일명으로 바꿔!

# 1) 데이터셋/로더 (학습과 동일한 전처리: Flatten 없음)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# 2) 디바이스 & 모델 생성 → 가중치 로드
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 학습 때 쓰던 아키텍처와 정확히 동일해야 함
model = KAN([28*28, 64, 10])  # base_activation 기본값이 SiLU면 이대로 OK
state = torch.load("mnist_kan_float.pth", map_location="cpu")
model.load_state_dict(state, strict=True)
model.to(device).eval()

# 3) 검증 루프 (루프에서만 flatten)
criterion = nn.CrossEntropyLoss()
val_loss_sum = 0.0
val_correct  = 0
val_total    = 0

with torch.inference_mode():
    for images, labels in tqdm(val_loader, desc="Validating", unit="batch"):
        images = images.view(-1, 28*28).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        val_loss_sum += loss.item() * images.size(0)
        val_correct  += (logits.argmax(1) == labels).sum().item()
        val_total    += images.size(0)

val_loss = val_loss_sum / val_total
val_acc  = val_correct / val_total
print(f"\n[LOAD&VAL] Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc*100:.2f}%")


In [ ]:
# bit 16 짜리 val 코드
# === Bit-spline KAN 로드 + Validation (학습 직후/별도 세션 둘 다 OK) ===
import torch, torch.nn as nn
import torchvision, torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# 0) 방금 덮어쓴 my_bit_spline 반영
import importlib, my_bit_spline
importlib.reload(my_bit_spline)
BitKAN = my_bit_spline.KAN

# 1) 데이터 (학습과 동일 전처리: Normalize만, flatten은 루프에서)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
valset = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)
valloader = DataLoader(valset, batch_size=64, shuffle=False, pin_memory=True)

# 2) 디바이스 & 모델 생성 (학습과 동일 아키텍처/하이퍼파라미터!)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ⚠️ PadeSiluApprox는 네가 학습 때 정의한 그 클래스(함수)로 그대로 넣어야 함
#    만약 현재 세션에 없다면, 먼저 그 정의 셀을 실행하거나 import 해와야 함.
bit_model = BitKAN([28*28, 64, 10], base_activation=PadeSiluApprox).to(device)

# 3) 가중치 로드 (mnist_kan_float.pth)
state = torch.load("mnist_kan_float.pth", map_location="cpu")
# DataParallel 저장 대비
if isinstance(state, dict) and "model" in state:
    state = state["model"]
if any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

bit_model.load_state_dict(state, strict=True)
bit_model.eval()

# 4) 검증 (루프에서만 flatten)
criterion = nn.CrossEntropyLoss()
val_loss_sum, val_correct, val_total = 0.0, 0, 0

with torch.inference_mode():
    for images, labels in tqdm(valloader, desc="Validating (bit-KAN)", unit="batch"):
        images = images.view(-1, 28*28).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = bit_model(images)
        loss = criterion(logits, labels)

        val_loss_sum += loss.item() * images.size(0)
        val_correct  += (logits.argmax(1) == labels).sum().item()
        val_total    += images.size(0)

val_loss = val_loss_sum / val_total
val_acc  = val_correct / val_total
print(f"\n[BIT-KAN] Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")


In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1) 준비: MNIST 검증용 데이터셋 & DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.Lambda(lambda x: x.view(-1))  # (1,28,28) -> (784,)
])

val_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# 2) Device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 3) 검증 수행 (진행바 추가)
criterion = nn.CrossEntropyLoss()
val_loss = 0.0
val_correct = 0
val_total = 0

# tqdm에 전체 배치 개수 전달하여 진척률 %, ETA 등을 보여줌
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Validating", unit="batch"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        val_correct += (predicted == labels).sum().item()
        val_total += images.size(0)

val_loss /= val_total
val_acc = val_correct / val_total
print(f"\nValidation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
###
# 1) (선택) float-KAN으로 학습 후 가중치 저장했다고 가정
# torch.save(model.state_dict(), "mnist_kan_float.pth")

# 2) bit-KAN 생성 + 가중치 로드
bit_model = BitKAN([28*28, 64, 10], base_activation=nn.SiLU).to(device)
bit_model.eval()

# 3) 핵심: 검증 루프 전에 모델 교체
model = bit_model
# 2) Device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
# bit_model 과 device 가 이미 정의되어 있고 eval() 인 상태라고 가정
import torch
import torch.nn as nn
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
val_loss_sum, val_correct, val_total = 0.0, 0, 0

bit_model.eval()
with torch.inference_mode():
    for images, labels in tqdm(valloader, desc="Validating (bit-KAN)", unit="batch"):
        images = images.view(-1, 28*28).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = bit_model(images)
        loss = criterion(logits, labels)

        val_loss_sum += loss.item() * images.size(0)
        val_correct  += (logits.argmax(1) == labels).sum().item()
        val_total    += images.size(0)

val_loss = val_loss_sum / val_total
val_acc  = val_correct / val_total
print(f"[bit-KAN] Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")


In [ ]:
print(type(model).__module__)           # <- 'my_bit_spline' 이어야 함
print(type(model.layers[0]).__module__) # <- 'my_bit_spline' 이어야 함

In [ ]:
import torch, torch.nn as nn

# 1) (선택) float-KAN 가중치 저장
torch.save(model.state_dict(), "mnist_kan_float.pth")

# 2) bit-KAN 생성 + 가중치 로드
from my_bit_spline import KAN as BitKAN
bit_model = BitKAN([28*28, 64, 10], base_activation=nn.SiLU).to(device)
bit_model.load_state_dict(torch.load("mnist_kan_float.pth", map_location=device), strict=False)
bit_model.eval()

# 3) 검증에 쓸 모델을 bit_model로 교체
model = bit_model

# 4) 진짜로 바뀌었는지 확인
print(type(model).__module__)           # <- 'my_bit_spline' 이어야 함
print(type(model.layers[0]).__module__) # <- 'my_bit_spline' 이어야 함


In [ ]:
from tqdm import tqdm
from my_bit_spline import KAN as BitKAN
import torch.nn as nn

bit_model = BitKAN([28*28, 64, 10], base_activation=nn.SiLU).to(device)
# ⚠️ 학습 가중치 로드했다면 여기에 load_state_dict 추가하세요.
# bit_model.load_state_dict(torch.load("mnist_kan_float.pth", map_location=device), strict=False)
bit_model.eval()

criterion = nn.CrossEntropyLoss()

val_loss_sum = 0.0
val_correct  = 0
val_total    = 0

with torch.no_grad():
    pbar = tqdm(valloader, desc="Validating (Bit-KAN)", unit="batch")
    for images, labels in pbar:
        images = images.view(-1, 28*28).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = bit_model(images)
        loss = criterion(logits, labels)

        # 누적
        val_loss_sum += loss.item() * images.size(0)
        val_correct  += (logits.argmax(1) == labels).sum().item()
        val_total    += images.size(0)

        # 진행바에 현재 평균 표시
        pbar.set_postfix(
            avg_loss = val_loss_sum / val_total,
            avg_acc  = val_correct / val_total
        )

val_loss = val_loss_sum / val_total
val_acc  = val_correct / val_total
print(f"[Bit-KAN] Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


In [ ]:
import torch
from PIL import Image
import torchvision.transforms as transforms

# Define the same transformations used during training
transform = transforms.Compose([
    transforms.Resize((28, 28)),  # Resize image to 28x28
    transforms.Grayscale(),       # Convert to grayscale if not already
    transforms.ToTensor(),        # Convert image to tensor
    transforms.Normalize((0.5,), (0.5,))  # Normalize
])

# Load and preprocess the image
image_path = './model/8hw.jpeg'
try:
    image = Image.open(image_path)
    image = transform(image)
    image = image.view(-1)  # Flatten the image to match the model's expected input
    image = image.unsqueeze(0)  # Add batch dimension

    print(f"Image shape after transformation: {image.shape}")
    image = image.to(device)
    # Make predictions
    with torch.no_grad():
        output = model(image)  # Forward pass
        _, predicted = torch.max(output, 1)  # Get the predicted digit

    # Convert tensor to digit
    predicted_digit = predicted.item()
    print(f'The predicted digit is: {predicted_digit}')

except FileNotFoundError as e:
    print(f"Error loading image: {e}")
except AssertionError as e:
    print(f"AssertionError: {e}")

# Print model layers for debugging
print("Model architecture:")
for name, layer in model.named_children():
    print(name, layer)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import torch.nn.functional as F

# 전체 예측 및 정답 수집
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

# ① 정밀도, 재현율, F1
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

# ② 혼동 행렬
conf_matrix = confusion_matrix(all_labels, all_preds)

print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")
print("Confusion Matrix:\n", conf_matrix)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 지표 이름 및 값
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
pade   = [96.97, 96.96, 96.93, 96.95]
silu   = [97.03, 97.03, 97.00, 97.01]
pade2r = [95.59, 95.61, 95.55, 95.54]  # 2차항 제거한 PADE

x = np.arange(len(metric_names))
width = 0.22

plt.figure(figsize=(9, 6))

# 막대 그래프
bars1 = plt.bar(x - width, silu, width, label='SiLU', color='skyblue')
bars2 = plt.bar(x, pade, width, label='PADE-SiLU', color='orange')
bars3 = plt.bar(x + width, pade2r, width, label='PADE (No $x^2$)', color='green')

# 축, 타이틀 등 설정
plt.ylabel('Score (%)')
plt.title('Validation Metrics Comparison: SiLU vs PADE vs PADE (No $x^2$)')
plt.xticks(x, metric_names)
plt.ylim(90, 102)
plt.legend()

# 수치 표시
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        plt.annotate(f'{height:.2f}',
                     xy=(bar.get_x() + bar.get_width() / 2, height),
                     xytext=(0, 3),
                     textcoords="offset points",
                     ha='center', va='bottom', fontsize=9)

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 지표 이름 및 값
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
silu   = [96.97, 96.96, 96.93, 96.95]
pade   = [97.05, 97.05, 97.02, 97.03]

x = np.arange(len(metric_names))
width = 0.22

plt.figure(figsize=(9, 6))

# 막대 그래프
bars1 = plt.bar(x - width, silu, width, label='SiLU', color='skyblue')
bars2 = plt.bar(x, pade, width, label='PADE-SiLU', color='orange')

# 축, 타이틀 등 설정
plt.ylabel('(%)')
plt.title('')
plt.xticks(x, metric_names)
plt.ylim(90, 102)
plt.legend()

# 수치 표시
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.annotate(f'{height:.2f}',
                     xy=(bar.get_x() + bar.get_width() / 2, height),
                     xytext=(0, 3),
                     textcoords="offset points",
                     ha='center', va='bottom', fontsize=9)

plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
def print_base_activations(model):
    """
    모델(예: KAN) 안에 있는 모든 KANLinear 레이어의 base_activation을 출력합니다.
    """
    for idx, layer in enumerate(model.layers):
        # layer는 KANLinear 인스턴스일 것
        activation_class_name = layer.base_activation.__class__.__name__
        print(f"Layer {idx} base_activation: {activation_class_name}")
print_base_activations(model)

In [ ]:
# 예: 간단한 Training + Validation 루프
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
import torchvision.transforms as transforms
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.Lambda(lambda x: x.view(-1))  # (1, 28, 28)->(784,)
])

valset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    transform=val_transform,
    download=True
)
valloader = torch.utils.data.DataLoader(valset, batch_size=64, shuffle=False)

# 변환을 적용한 trainset 생성
trainset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

# trainloader를 새로 만들어야 함
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)

for epoch in range(num_epochs):
    model.train()  # 학습 모드
    running_loss = 0.0
    running_correct = 0
    running_samples = 0

    for images, labels in trainloader:  # trainloader는 학습용 DataLoader
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # 학습 상황 기록
        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        _, predicted = torch.max(outputs, 1)
        running_correct += (predicted == labels).sum().item()
        running_samples += batch_size

    train_loss = running_loss / running_samples
    train_acc = running_correct / running_samples

    # === Validation ===
    val_loss, val_acc = validate(model, valloader, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


In [ ]:
images, labels = next(iter(trainloader))
print(images.shape)
images_val, labels_val = next(iter(valloader))
print(images_val.shape)  # 기대: [64, 784]

In [ ]:
import torch

# B-spline 계산을 위한 함수 정의
def b_spline_basis(i, p, t, knots):
    if p == 0:
        return torch.where((knots[i] <= t) & (t < knots[i + 1]), torch.tensor(1.0, dtype=torch.float64), torch.tensor(0.0, dtype=torch.float64))

    denom1 = knots[i + p] - knots[i]
    denom2 = knots[i + p + 1] - knots[i + 1]

    term1 = torch.tensor(0.0, dtype=torch.float64)
    term2 = torch.tensor(0.0, dtype=torch.float64)

    if denom1 != 0:
        term1 = ((t - knots[i]) / denom1) * b_spline_basis(i, p - 1, t, knots)
    if denom2 != 0:
        term2 = ((knots[i + p + 1] - t) / denom2) * b_spline_basis(i + 1, p - 1, t, knots)

    return term1 + term2

# SiLU 함수 정의
def silu(x):
    return x / (1 + torch.exp(-x))

# phi(x) 계산 함수
def phi(x, knots, coefficients, wb, ws, p=3):
    n = len(knots) - p - 2
    spline = torch.tensor(0.0, dtype=torch.float64)
    for i in range(n + 1):
        basis = b_spline_basis(i, p, x, knots)
        spline += coefficients[i] * basis

    b_x = silu(x)
    return wb * b_x + ws * spline

# 파라미터 설정 (각 노드별 계수)
knots = torch.tensor([-3.0, -2.3333333, -1.6666667, -1.0, -0.3333333, 0.3333333, 1.0, 1.6666667, 2.3333333, 3.0], dtype=torch.float64)
#knots = torch.tensor([-2.5, -2.0, -1.5, -1.0, -0.333333333333333, 0.333333333333333, 1.0, 1.5, 2.0, 2.5], dtype=torch.float64)

# Node (0, 0, 0) 계수
wb_0 = torch.tensor(1.17158114910125732422, dtype=torch.float64)
ws_0 = torch.tensor(1.01455640792846679688, dtype=torch.float64)
coefficients_0 = torch.tensor([-1.2948788404464722, -0.6209051012992859, -0.10060922801494598, 0.17169933021068573, 0.19301898777484894, 0.05289100110530853], dtype=torch.float64)

#wb_0 = torch.tensor(1.17)
#ws_0 = torch.tensor(1.014)
#coefficients_0 = torch.tensor([-1.2948, -0.620, -0.100, 0.171, 0.193, 0.052])

# Node (0, 1, 0) 계수
wb_1 = torch.tensor(1.05469250679016113281, dtype=torch.float64)
ws_1 = torch.tensor(1.04214644432067871094, dtype=torch.float64)
coefficients_1 = torch.tensor([-1.3457839488983154, -0.6817834377288818, -0.1634548008441925, 0.1335010826587677, 0.21377956867218018, 0.15929093956947327], dtype=torch.float64)

#wb_1 = torch.tensor(1.054)
#ws_1 = torch.tensor(1.042)
#coefficients_1 = torch.tensor([-1.345, -0.6, -0.163, 0.133, 0.213, 0.159])



# 테스트: 2D 입력 [x1, x2]에 대해 phi_0(x1) + phi_1(x2) 계산
def main():
    test_x_pairs = [
        (0.5, 0.5),
        (1.0, 1.0),
        (-0.7, 0.2),
        (0.2, 0.1),
        (3.0, 4.0)
    ]

    for x1, x2 in test_x_pairs:
        x1_tensor = torch.tensor(x1, dtype=torch.float64)
        x2_tensor = torch.tensor(x2, dtype=torch.float64)
        # 첫 번째 노드: phi_0(x1)
        phi_x1 = phi(x1_tensor, knots, coefficients_0, wb_0, ws_0)
        # 두 번째 노드: phi_1(x2)
        phi_x2 = phi(x2_tensor, knots, coefficients_1, wb_1, ws_1)
        # 최종 출력: phi_0(x1) + phi_1(x2)
        y = phi_x1 + phi_x2
        print(f"x1 = {x1:.1f}, x2 = {x2:.1f}, phi_0(x1) + phi_1(x2) = {y.item():.6f}")

if __name__ == "__main__":
    main()

In [ ]:
print("\n=== Efficient KAN Parameters by Node ===")

# Efficient KAN은 width 정보 대신, layers 리스트와 각 layer 안의 파라미터를 직접 조회해야 함
for layer_idx, layer in enumerate(model.layers):
    in_features = layer.in_features
    out_features = layer.out_features
    print(f"\nLayer {layer_idx} (Input: {in_features}, Output: {out_features}):")

    for in_node in range(in_features):
        for out_node in range(out_features):
            try:
                # B-spline 계수 (ci)
                coeffs = layer.spline_weight[out_node, in_node]  # shape: [grid_size + spline_order]
                # w_b (base_weight)
                w_b = layer.base_weight[out_node, in_node]
                # w_s (scale_sp)
                if layer.enable_standalone_scale_spline:
                    w_s = layer.spline_scaler[out_node, in_node]
                else:
                    w_s = torch.tensor(1.0)

                print(f"  Node ({layer_idx}, {in_node}, {out_node}):")
                print(f"    w_b: {w_b.item():.20f}")
                print(f"    w_s: {w_s.item():.20f}")
                print(f"    c_i: {[f'{c.item():.20f}' for c in coeffs]}")
            except Exception as e:
                print(f"    Error accessing Node ({layer_idx}, {in_node}, {out_node}): {e}")


#### 다양한근사 확인용


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd

# Ground truth: exp(-x)
def true_exp_neg(x):
    return torch.exp(-x)

# Padé 근사 정의 (2차부터 5차까지)
class PadeExpApprox2(nn.Module):
    def forward(self, x):
        num = 1 - 0.5 * x + (x ** 2) / 12
        den = 1 + 0.5 * x + (x ** 2) / 12
        return num / den

class PadeExpApprox3(nn.Module):
    def forward(self, x):
        num = 1 - 0.5 * x + (x ** 2) / 10 - (x ** 3) / 120
        den = 1 + 0.5 * x + (x ** 2) / 10 + (x ** 3) / 120
        return num / den

class PadeExpApprox4(nn.Module):
    def forward(self, x):
        num = 1 - 0.5 * x + (x ** 2) / 9 - (x ** 3) / 162 + (x ** 4) / 5184
        den = 1 + 0.5 * x + (x ** 2) / 9 + (x ** 3) / 162 + (x ** 4) / 5184
        return num / den

class PadeExpApprox5(nn.Module):
    def forward(self, x):
        num = 1 - 0.5 * x + (x ** 2) / 12 - (x ** 3) / 120 + (x ** 4) / 1440 - (x ** 5) / 86400
        den = 1 + 0.5 * x + (x ** 2) / 12 + (x ** 3) / 120 + (x ** 4) / 1440 + (x ** 5) / 86400
        return num / den

class PadeExpApprox13(nn.Module):
    def forward(self, x):
        num = 1 - (x / 4)
        den = 1 + (x / 4) + (x ** 2) / 8 + (x ** 3) / 192
        return num / den

class PadeExpApprox24(nn.Module):
    def forward(self, x):
        num = 1 - (x / 3) + (x ** 2) / 18
        den = 1 + (x / 3) + (x ** 2) / 18 + (x ** 3) / 162 + (x ** 4) / 3888
        return num / den

class PadeExpApprox26(nn.Module):
    def forward(self, x):
        num = 1 - 0.5 * x + (x ** 2) / 12
        den = 1 + 0.5 * x + (x ** 2) / 12 + (x ** 3) / 120 + (x ** 4) / 1680 + (x ** 5) / 30240 + (x ** 6) / 665280
        return num / den

class PadeExpApprox34(nn.Module):  # Padé [3/4]
    def forward(self, x):
        num = 1 - 0.5 * x + 0.1 * x**2 - (1/120) * x**3
        den = 1 + 0.5 * x + 0.1 * x**2 + (1/120) * x**3 + (1/1680) * x**4
        return num / den

x = torch.linspace(-1, 1, 5000)
true_vals = true_exp_neg(x)

# 모델 리스트
models = {
    "Padé 1/3": PadeExpApprox13(),
    "Padé 2/4": PadeExpApprox24(),
    "Padé 2/6": PadeExpApprox26(),
    "Padé 2nd": PadeExpApprox2(),
    "Padé 3rd": PadeExpApprox3(),
    "Padé 4th": PadeExpApprox4(),
    "Padé 5th": PadeExpApprox5(),
    "Padé 3/4": PadeExpApprox34()
}

# 결과 저장
results = {}
for name, model in models.items():
    approx = model(x)
    error = true_vals - approx
    mse = torch.mean(error ** 2).item()
    mae = torch.mean(torch.abs(error)).item()
    maxe = torch.max(torch.abs(error)).item()
    results[name] = {
        "approx": approx.detach().numpy(),
        "MSE": mse,
        "MAE": mae,
        "Max Error": maxe
    }

# 📊 시각화
plt.figure(figsize=(12, 6))
plt.plot(x, true_vals, label="True exp(-x)", linewidth=2)
for name in models.keys():
    plt.plot(x, results[name]["approx"], label=name)
plt.title("exp(-x) vs Padé Approximations")
plt.xlabel("x")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()

# 📋 오차 지표 정리
error_df = pd.DataFrame({
    "Model": list(results.keys()),
    "MSE": [results[m]["MSE"] for m in results],
    "MAE": [results[m]["MAE"] for m in results],
    "Max Error": [results[m]["Max Error"] for m in results],
})
print(error_df)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

# ✅ Ground truth: SiLU = x * sigmoid(x)
def true_silu(x):
    return F.silu(x)

# ✅ Padé 기반 SiLU 근사 정의 (exp(-x) 근사 이용)
class PadeSiluApprox2(nn.Module):
    def forward(self, x):
        exp_approx = (1 - 0.5 * x + (x ** 2) / 12) / (1 + 0.5 * x + (x ** 2) / 12)
        return x / (1 + exp_approx)

class PadeSiluApprox3(nn.Module):
    def forward(self, x):
        exp_approx = (1 - 0.5 * x + (x ** 2) / 10 - (x ** 3) / 120) / \
                     (1 + 0.5 * x + (x ** 2) / 10 + (x ** 3) / 120)
        return x / (1 + exp_approx)

class PadeSiluApprox4(nn.Module):
    def forward(self, x):
        exp_approx = (1 - 0.5 * x + (x ** 2) / 9 - (x ** 3) / 162 + (x ** 4) / 5184) / \
                     (1 + 0.5 * x + (x ** 2) / 9 + (x ** 3) / 162 + (x ** 4) / 5184)
        return x / (1 + exp_approx)

class PadeSiluApprox5(nn.Module):
    def forward(self, x):
        exp_approx = (1 - 0.5 * x + (x ** 2) / 10 - (x ** 3) / 120 + (x ** 4) / 1440 - (x ** 5) / 86400) / \
                     (1 + 0.5 * x + (x ** 2) / 10 + (x ** 3) / 120 + (x ** 4) / 1440 + (x ** 5) / 86400)
        return x / (1 + exp_approx)

class PadeSiluApprox13(nn.Module):
    def forward(self, x):
        num = 1 - (x / 4)
        den = 1 + (x / 4) + (x ** 2) / 8 + (x ** 3) / 192
        exp_approx = num / den
        return x / (1 + exp_approx)

class PadeSiluApprox24(nn.Module):
    def forward(self, x):
        num = 1 - (x / 3) + (x ** 2) / 18
        den = 1 + (x / 3) + (x ** 2) / 18 + (x ** 3) / 162 + (x ** 4) / 3888
        exp_approx = num / den
        return x / (1 + exp_approx)

class PadeSiluApprox26(nn.Module):
    def forward(self, x):
        num = 1 - 0.5 * x + (x ** 2) / 12
        den = 1 + 0.5 * x + (x ** 2) / 12 + (x ** 3) / 120 + (x ** 4) / 1680 + (x ** 5) / 30240 + (x ** 6) / 665280
        exp_approx = num / den
        return x / (1 + exp_approx)

# ✅ 입력 정의
x = torch.linspace(-5, 5, 1000)
true_vals = true_silu(x)

# ✅ 모델 정의
models = {
    "Padé SiLU 1/3": PadeSiluApprox13(),
    "Padé SiLU 2/4": PadeSiluApprox24(),
    "Padé SiLU 2/6": PadeSiluApprox26(),
    "Padé SiLU 2nd": PadeSiluApprox2(),
    "Padé SiLU 3rd": PadeSiluApprox3(),
    "Padé SiLU 4th": PadeSiluApprox4(),
    "Padé SiLU 5th": PadeSiluApprox5()
}

# ✅ 결과 저장
results = {}
for name, model in models.items():
    approx = model(x)
    error = true_vals - approx
    mse = torch.mean(error ** 2).item()
    mae = torch.mean(torch.abs(error)).item()
    maxe = torch.max(torch.abs(error)).item()
    results[name] = {
        "approx": approx.detach().numpy(),
        "MSE": mse,
        "MAE": mae,
        "Max Error": maxe
    }

# ✅ 시각화
plt.figure(figsize=(12, 6))
plt.plot(x, true_vals, label="True SiLU", linewidth=2)
for name in models.keys():
    plt.plot(x, results[name]["approx"], label=name)
plt.title("SiLU vs Padé Approximations")
plt.xlabel("x")
plt.ylabel("Value")
plt.legend()
plt.grid(True)
plt.show()

# ✅ 오차 지표 출력
error_df = pd.DataFrame({
    "Model": list(results.keys()),
    "MSE": [results[m]["MSE"] for m in results],
    "MAE": [results[m]["MAE"] for m in results],
    "Max Error": [results[m]["Max Error"] for m in results],
})
print(error_df)


#### 비트수 통계

In [ ]:
import math, torch, torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class RunningStats:
    n: int = 0
    s: float = 0.0
    ss: float = 0.0
    vmin: float = float("inf")
    vmax: float = float("-inf")
    amax: float = 0.0

    def update_tensor(self, t: torch.Tensor):
        # t: 1D or any shape; we flatten
        t = t.reshape(-1).to(torch.float32).cpu()
        if t.numel() == 0:
            return
        self.n   += t.numel()
        self.s   += t.sum().item()
        self.ss  += (t*t).sum().item()
        tmin      = t.min().item()
        tmax      = t.max().item()
        tabsmax   = t.abs().max().item()
        self.vmin = min(self.vmin, tmin)
        self.vmax = max(self.vmax, tmax)
        self.amax = max(self.amax, tabsmax)

    def summary(self):
        mean = self.s / self.n if self.n else 0.0
        var  = max(self.ss / self.n - mean*mean, 0.0) if self.n else 0.0
        std  = math.sqrt(var)
        return dict(mean=mean, std=std, min=self.vmin, max=self.vmax, absmax=self.amax, count=self.n)


In [ ]:
from tqdm import tqdm

def inspect_hidden_outputs_stats(model, dataloader, device, limit_batches=None):
    """
    각 KANLinear 레이어의 '출력'(batch,out_features)의 전 구간 통계를 모읍니다.
    """
    model.eval()
    # KANLinear 레이어들만 대상으로 통계를 준비
    layers = [m for m in model.modules() if m.__class__.__name__ == "KANLinear"]
    stats  = [RunningStats() for _ in layers]
    handles = []

    # 후크로 레이어 출력 캡처
    def make_hook(i):
        def hook(module, inp, out):
            stats[i].update_tensor(out)
        return hook

    for i, L in enumerate(layers):
        handles.append(L.register_forward_hook(make_hook(i)))

    # 순회
    with torch.no_grad():
        for b_ix, (images, *rest) in enumerate(tqdm(dataloader, desc="Collect hidden stats", unit="batch")):
            x = images.to(device)
            _ = model(x)  # forward를 흘려 보내면 후크가 출력들을 수집
            if limit_batches is not None and (b_ix+1) >= limit_batches:
                break

    # 후크 제거
    for h in handles:
        h.remove()

    # 출력
    print("\n[Hidden layer output stats]")
    for i, rs in enumerate(stats):
        sm = rs.summary()
        print(f"  Layer {i}: mean={sm['mean']:.6f}, std={sm['std']:.6f}, "
              f"min={sm['min']:.6f}, max={sm['max']:.6f}, absmax={sm['absmax']:.6f}, count={sm['count']}")
    return stats


In [ ]:
def kanlinear_forward_with_parts(layer, x):
    """
    KANLinear의 forward를 base/spline으로 분해해 반환
    base_out = W_base * base_activation(x)
    spline_out = W_spline * B_splines(x)
    total = base_out + spline_out
    """
    x2 = x.reshape(-1, layer.in_features)
    base_out = F.linear(layer.base_activation(x2), layer.base_weight)                 # (B, out)
    bs       = layer.b_splines(x2).view(x2.size(0), -1)                               # (B, in*(grid_size+spline_order))
    spline_out = F.linear(bs, layer.scaled_spline_weight.view(layer.out_features, -1))# (B, out)
    total = base_out + spline_out
    return base_out, spline_out, total

def inspect_base_spline_stats(model, dataloader, device, limit_batches=None):
    """
    각 레이어별 base_out / spline_out / total의 통계를 모읍니다.
    """
    model.eval()
    layers = [m for m in model.modules() if m.__class__.__name__ == "KANLinear"]
    base_stats   = [RunningStats() for _ in layers]
    spline_stats = [RunningStats() for _ in layers]
    total_stats  = [RunningStats() for _ in layers]

    with torch.no_grad():
        for b_ix, (images, *rest) in enumerate(tqdm(dataloader, desc="Collect base/spline stats", unit="batch")):
            x = images.to(device)
            for li, L in enumerate(layers):
                b_out, s_out, t_out = kanlinear_forward_with_parts(L, x)
                base_stats[li].update_tensor(b_out)
                spline_stats[li].update_tensor(s_out)
                total_stats[li].update_tensor(t_out)
                x = t_out  # 다음 레이어 입력은 total
            if limit_batches is not None and (b_ix+1) >= limit_batches:
                break

    print("\n[Per-layer base/spline/total stats]")
    for i in range(len(layers)):
        sb = base_stats[i].summary()
        ss = spline_stats[i].summary()
        st = total_stats[i].summary()
        print(f"  Layer {i}:")
        print(f"    base   : mean={sb['mean']:.6f}, std={sb['std']:.6f}, min={sb['min']:.6f}, max={sb['max']:.6f}, absmax={sb['absmax']:.6f}")
        print(f"    spline : mean={ss['mean']:.6f}, std={ss['std']:.6f}, min={ss['min']:.6f}, max={ss['max']:.6f}, absmax={ss['absmax']:.6f}")
        print(f"    total  : mean={st['mean']:.6f}, std={st['std']:.6f}, min={st['min']:.6f}, max={st['max']:.6f}, absmax={st['absmax']:.6f}")
    return base_stats, spline_stats, total_stats


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

# 전체를 다 돌리면 오래 걸릴 수 있으니, 먼저 50배치 정도로 대략 분포 확인
_ = inspect_hidden_outputs_stats(model, val_loader, device, limit_batches=50)

# base / spline 기여분도 확인
_ = inspect_base_spline_stats(model, val_loader, device, limit_batches=50)

# 충분히 빠르면 limit_batches=None으로 전체 데이터셋 통계 수집
_ = inspect_hidden_outputs_stats(model, val_loader, device, limit_batches=None)
_ = inspect_base_spline_stats(model, val_loader, device, limit_batches=None)


In [ ]:
import torch
import math

def _tensor_stats(t: torch.Tensor):
    # 파라미터 텐서 통계 (float32로 변환 후 CPU에서 계산)
    t = t.detach()
    dev = str(t.device)
    dt  = str(t.dtype)
    num = t.numel()
    if num == 0:
        return {
            "device": dev, "dtype": dt, "numel": num, "shape": list(t.shape),
            "mean": None, "std": None, "min": None, "max": None, "absmax": None
        }
    tc = t.float().view(-1).cpu()
    mean = tc.mean().item()
    # 분산이 음수로 내려가는 수치오차 방지
    var  = max(tc.pow(2).mean().item() - mean*mean, 0.0)
    std  = math.sqrt(var)
    return {
        "device": dev,
        "dtype": dt,
        "numel": num,
        "shape": list(t.shape),
        "mean": mean,
        "std": std,
        "min": tc.min().item(),
        "max": tc.max().item(),
        "absmax": tc.abs().max().item(),
    }

def report_all_param_stats(model: torch.nn.Module, include_buffers: bool = False):
    """
    모델의 모든 파라미터(named_parameters) 통계를 출력.
    include_buffers=True면 register_buffer로 등록된 버퍼(named_buffers)도 함께 출력.
    """
    print("\n========== Parameter Stats ==========")
    total_params = 0
    for name, p in model.named_parameters():
        s = _tensor_stats(p)
        total_params += s["numel"]
        print(f"[PARAM] {name:40s} shape={s['shape']} dtype={s['dtype']} device={s['device']} numel={s['numel']}")
        print(f"        mean={s['mean']:.6g}, std={s['std']:.6g}, "
              f"min={s['min']:.6g}, max={s['max']:.6g}, absmax={s['absmax']:.6g}")
    print(f"Total param count: {total_params}")

    if include_buffers:
        print("\n========== Buffer Stats ==========")
        for name, b in model.named_buffers():
            s = _tensor_stats(b)
            print(f"[BUFFER] {name:40s} shape={s['shape']} dtype={s['dtype']} device={s['device']} numel={s['numel']}")
            # 버퍼는 비가중치일 수 있으니 None 체크
            if s["mean"] is not None:
                print(f"         mean={s['mean']:.6g}, std={s['std']:.6g}, "
                      f"min={s['min']:.6g}, max={s['max']:.6g}, absmax={s['absmax']:.6g}")
import torch

def _tensor_statss(t: torch.Tensor):
    t = t.detach().float().cpu()
    return {
        "shape": list(t.shape),
        "mean": t.mean().item(),
        "std":  t.std(unbiased=False).item(),
        "min":  t.min().item(),
        "max":  t.max().item(),
        "absmax": t.abs().max().item(),
    }

def report_kan_param_stats(model):
    """
    학습 완료된 기본 KAN 모델의 파라미터 통계를 레이어별로 출력.
    - base_weight (W_b)
    - spline_weight (W_s)
    - scaled_spline_weight (W_s,eff)
    - 각 C_k (k=1..Ks)별 통계 (raw/eff)
    - F.linear에 실제 들어가는 행렬(평탄화) 통계
    - spline_scaler 통계(있을 때)
    """
    # KANLinear만 골라내기
    layers = [m for m in model.modules() if m.__class__.__name__ == "KANLinear"]
    if not layers:
        print("KANLinear 레이어를 찾지 못했습니다. 올바른 KAN 모델인지 확인하세요.")
        return

    print("\n========== KAN Parameter Stats ==========")
    for li, L in enumerate(layers):
        print(f"\n[Layer {li}] in={L.in_features}, out={L.out_features}, "
              f"grid_size={L.grid_size}, spline_order={L.spline_order}")
        Ks = L.grid_size + L.spline_order

        # 1) base weight
        wb = L.base_weight
        s_wb = _tensor_stats(wb)
        print(f"  W_b (base_weight) {s_wb['shape']}: "
              f"mean={s_wb['mean']:.6f}, std={s_wb['std']:.6f}, "
              f"min={s_wb['min']:.6f}, max={s_wb['max']:.6f}, absmax={s_wb['absmax']:.6f}")

        # 2) spline weights (raw / effective)
        ws_raw = L.spline_weight
        s_ws_raw = _tensor_statss(ws_raw)
        print(f"  W_s (spline_weight raw) {s_ws_raw['shape']}: "
              f"mean={s_ws_raw['mean']:.6f}, std={s_ws_raw['std']:.6f}, "
              f"min={s_ws_raw['min']:.6f}, max={s_ws_raw['max']:.6f}, absmax={s_ws_raw['absmax']:.6f}")

        ws_eff = L.scaled_spline_weight  # property
        s_ws_eff = _tensor_statss(ws_eff)
        print(f"  W_s,eff (scaled_spline_weight) {s_ws_eff['shape']}: "
              f"mean={s_ws_eff['mean']:.6f}, std={s_ws_eff['std']:.6f}, "
              f"min={s_ws_eff['min']:.6f}, max={s_ws_eff['max']:.6f}, absmax={s_ws_eff['absmax']:.6f}")

        # 3) C_k별 통계 (k = 1..Ks) — (out,in) over axes
        print("  Per-coefficient (C_k) stats over (out,in):")
        for k in range(Ks):
            s_raw_k = _tensor_statss(ws_raw[..., k])
            s_eff_k = _tensor_statss(ws_eff[..., k])
            print(f"    C_{k+1:02d}: raw(mean={s_raw_k['mean']:.6f}, std={s_raw_k['std']:.6f}, "
                  f"min={s_raw_k['min']:.6f}, max={s_raw_k['max']:.6f}, absmax={s_raw_k['absmax']:.6f}) | "
                  f"eff(mean={s_eff_k['mean']:.6f}, std={s_eff_k['std']:.6f}, "
                  f"min={s_eff_k['min']:.6f}, max={s_eff_k['max']:.6f}, absmax={s_eff_k['absmax']:.6f})")

        # 4) 실제 F.linear에 들어가는 행렬 통계
        wb_mat = wb                 # (out, in)
        ws_mat = ws_eff.view(L.out_features, -1)  # (out, in*Ks)
        s_wb_mat = _tensor_statss(wb_mat)
        s_ws_mat = _tensor_statss(ws_mat)
        print(f"  [F.linear matrices]")
        print(f"    base mat (out,in) {s_wb_mat['shape']}: "
              f"mean={s_wb_mat['mean']:.6f}, std={s_wb_mat['std']:.6f}, "
              f"min={s_wb_mat['min']:.6f}, max={s_wb_mat['max']:.6f}, absmax={s_wb_mat['absmax']:.6f}")
        print(f"    spline mat (out,in*Ks) {s_ws_mat['shape']}: "
              f"mean={s_ws_mat['mean']:.6f}, std={s_ws_mat['std']:.6f}, "
              f"min={s_ws_mat['min']:.6f}, max={s_ws_mat['max']:.6f}, absmax={s_ws_mat['absmax']:.6f}")

        # 5) spline_scaler(있다면)
        if hasattr(L, "spline_scaler"):
            ss = _tensor_stats(L.spline_scaler)
            print(f"  spline_scaler {ss['shape']}: "
                  f"mean={ss['mean']:.6f}, std={ss['std']:.6f}, "
                  f"min={ss['min']:.6f}, max={ss['max']:.6f}, absmax={ss['absmax']:.6f}")

        # (선택) grid 버퍼도 보고 싶다면 주석 해제
        # g = _tensor_stats(L.grid)
        # print(f"  grid {g['shape']}: min={g['min']:.6f}, max={g['max']:.6f}")

    print("\n=========================================\n")


In [ ]:
model.eval()  # 선택
report_all_param_stats(model, include_buffers=True)  # 버퍼(grid 등)까지 보고 싶으면 True
model.to('cpu').eval()  # 어디서든 동작하지만, 출력만 볼 거면 CPU로 빼는 게 안전
report_kan_param_stats(model)


#### **비트**

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class PadeSiluApprox1(nn.Module):
    def __init__(self):
        super(PadeSiluApprox1, self).__init__()
        # 상수들을 클래스 속성으로 정의 (20-bit fixed-point)
        self.const_1 = self.float_to_fixed20(1.0)
        self.const_1_2 = self.float_to_fixed20(1.0 / 2.0)
        self.const_1_10 = self.float_to_fixed20(1.0 / 10.0)
        self.const_1_120 = self.float_to_fixed20(1.0 / 120.0)
        self.const_2 = self.float_to_fixed20(2.0)
        self.const_1_020 = self.float_to_fixed20(1.020)
        self.const_1_040 = self.float_to_fixed20(1.040)
        self.const_1_060 = self.float_to_fixed20(1.060)
        self.const_1_080 = self.float_to_fixed20(1.080)
        self.const_0_999 = self.float_to_fixed20(0.999)
        self.const_0_980 = self.float_to_fixed20(0.980)
        self.const_0_961 = self.float_to_fixed20(0.961)
        self.const_0_943 = self.float_to_fixed20(0.943)
        self.const_0_925 = self.float_to_fixed20(0.925)

    def float_to_fixed20(self, value: float) -> str:
        SCALE = 2**16
        MAX_INT = 2**19 - 1
        MIN_INT = -2**19

        scaled = int(round(value * SCALE))
        if not (MIN_INT <= scaled <= MAX_INT):
            raise ValueError(f"Value {value} out of range for 20-bit fixed-point format.")

        if scaled >= 0:
            bin_str = format(scaled, '020b')
        else:
            bin_str = format((1 << 20) + scaled, '020b')
        return bin_str

    def fixed20_to_float(self, bitstring: str) -> float:
        if len(bitstring) != 20:
            raise ValueError("Bitstring must be exactly 20 bits long.")

        int_value = int(bitstring, 2)
        if bitstring[0] == '1':
            int_value -= (1 << 20)
        return int_value / (2**16)

    def add_fixed20(self, a: str, b: str) -> str:
        a_int = int(a, 2)
        b_int = int(b, 2)
        if a[0] == '1':
            a_int -= 1 << 20
        if b[0] == '1':
            b_int -= 1 << 20

        result = a_int + b_int
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def sub_fixed20(self, a: str, b: str) -> str:
        a_int = int(a, 2)
        b_int = int(b, 2)
        if a[0] == '1':
            a_int -= 1 << 20
        if b[0] == '1':
            b_int -= 1 << 20

        result = a_int - b_int
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def mul_fixed20(self, a: str, b: str) -> str:
        a_int = int(a, 2)
        b_int = int(b, 2)
        if a[0] == '1':
            a_int -= 1 << 20
        if b[0] == '1':
            b_int -= 1 << 20

        result = (a_int * b_int) >> 16
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def square_fixed20(self, a: str) -> str:
        a_int = int(a, 2)
        if a[0] == '1':
            a_int -= 1 << 20

        result = (a_int * a_int) >> 16
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def cube_fixed20(self, a: str) -> str:
        a_int = int(a, 2)
        if a[0] == '1':
            a_int -= 1 << 20

        result = (a_int * a_int * a_int) >> 32
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def get_initial_guess(self, denom_val: str) -> str:
        d = self.fixed20_to_float(denom_val)
        if 1.0 <= d < 1.020:
            return self.const_0_999
        elif 1.020 <= d < 1.040:
            return self.const_0_980
        elif 1.040 <= d < 1.060:
            return self.const_0_961
        elif 1.060 <= d < 1.080:
            return self.const_0_943
        else:
            return self.const_0_925

    def newton_division(self, num: str, denom: str, iterations: int) -> str:
        x_n = self.get_initial_guess(denom)
        for _ in range(iterations):
            x_n = self.mul_fixed20(x_n, self.sub_fixed20(self.const_2, self.mul_fixed20(denom, x_n)))
        return self.mul_fixed20(num, x_n)

    def custom_silu(self, x: str, iterations: int = 2) -> str:
        first = self.const_1
        second = self.mul_fixed20(self.const_1_2, x)
        third = self.mul_fixed20(self.const_1_10, self.square_fixed20(x))
        fourth = self.mul_fixed20(self.const_1_120, self.cube_fixed20(x))

        first_numerator = self.add_fixed20(first, self.add_fixed20(second, self.add_fixed20(third, fourth)))
        numerator = self.mul_fixed20(x, first_numerator)
        denominator = self.add_fixed20(self.mul_fixed20(self.const_1_10, self.square_fixed20(x)), self.const_1)

        approx_div = self.newton_division(numerator, denominator, iterations)
        return self.mul_fixed20(self.const_1_2, approx_div)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        results = []
        for x_i in x.flatten():
            x_fixed = self.float_to_fixed20(x_i.item())
            result_fixed = self.custom_silu(x_fixed, iterations=2)
            result_float = self.fixed20_to_float(result_fixed)
            results.append(result_float)
        return torch.tensor(results, device=x.device).view_as(x)


In [ ]:
import torch
import torch.nn as nn
import numpy as np

class PadeSiluApprox1(nn.Module):
    def __init__(self):
        super(PadeSiluApprox1, self).__init__()
        # 상수들을 클래스 속성으로 정의 (20-bit fixed-point)
        self.const_1 = self.float_to_fixed20(1.0)
        self.const_1_2 = self.float_to_fixed20(1.0 / 2.0)
        self.const_1_10 = self.float_to_fixed20(1.0 / 10.0)
        self.const_1_120 = self.float_to_fixed20(1.0 / 120.0)
        self.const_2 = self.float_to_fixed20(2.0)
        self.const_1_020 = self.float_to_fixed20(1.020)
        self.const_1_040 = self.float_to_fixed20(1.040)
        self.const_1_060 = self.float_to_fixed20(1.060)
        self.const_1_080 = self.float_to_fixed20(1.080)
        self.const_0_999 = self.float_to_fixed20(0.999)
        self.const_0_980 = self.float_to_fixed20(0.980)
        self.const_0_961 = self.float_to_fixed20(0.961)
        self.const_0_943 = self.float_to_fixed20(0.943)
        self.const_0_925 = self.float_to_fixed20(0.925)

    def float_to_fixed20(self, value: float) -> str:
        SCALE = 2**16
        MAX_INT = 2**19 - 1
        MIN_INT = -2**19

        scaled = int(round(value * SCALE))
        if not (MIN_INT <= scaled <= MAX_INT):
            raise ValueError(f"Value {value} out of range for 20-bit fixed-point format.")

        if scaled >= 0:
            bin_str = format(scaled, '020b')
        else:
            bin_str = format((1 << 20) + scaled, '020b')
        return bin_str

    def fixed20_to_float(self, bitstring: str) -> float:
        if len(bitstring) != 20:
            raise ValueError("Bitstring must be exactly 20 bits long.")

        int_value = int(bitstring, 2)
        if bitstring[0] == '1':
            int_value -= (1 << 20)
        return int_value / (2**16)

    def add_fixed20(self, a: str, b: str) -> str:
        a_int = int(a, 2)
        b_int = int(b, 2)
        if a[0] == '1':
            a_int -= 1 << 20
        if b[0] == '1':
            b_int -= 1 << 20

        result = a_int + b_int
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def sub_fixed20(self, a: str, b: str) -> str:
        a_int = int(a, 2)
        b_int = int(b, 2)
        if a[0] == '1':
            a_int -= 1 << 20
        if b[0] == '1':
            b_int -= 1 << 20

        result = a_int - b_int
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def mul_fixed20(self, a: str, b: str) -> str:
        a_int = int(a, 2)
        b_int = int(b, 2)
        if a[0] == '1':
            a_int -= 1 << 20
        if b[0] == '1':
            b_int -= 1 << 20

        result = (a_int * b_int) >> 16
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def square_fixed20(self, a: str) -> str:
        a_int = int(a, 2)
        if a[0] == '1':
            a_int -= 1 << 20

        result = (a_int * a_int) >> 16
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def cube_fixed20(self, a: str) -> str:
        a_int = int(a, 2)
        if a[0] == '1':
            a_int -= 1 << 20

        result = (a_int * a_int * a_int) >> 32
        if result < 0:
            result = (1 << 20) + result
        return format(result & ((1 << 20) - 1), '020b')

    def get_initial_guess(self, denom_val: str) -> str:
        d = self.fixed20_to_float(denom_val)
        if 1.0 <= d < 1.020:
            return self.const_0_999
        elif 1.020 <= d < 1.040:
            return self.const_0_980
        elif 1.040 <= d < 1.060:
            return self.const_0_961
        elif 1.060 <= d < 1.080:
            return self.const_0_943
        else:
            return self.const_0_925

    def newton_division(self, num: str, denom: str, iterations: int) -> str:
        x_n = self.get_initial_guess(denom)
        for _ in range(iterations):
            x_n = self.mul_fixed20(x_n, self.sub_fixed20(self.const_2, self.mul_fixed20(denom, x_n)))
        return self.mul_fixed20(num, x_n)

    def custom_silu(self, x: str, iterations: int = 2) -> str:
        first = self.const_1
        second = self.mul_fixed20(self.const_1_2, x)
        third = self.mul_fixed20(self.const_1_10, self.square_fixed20(x))
        fourth = self.mul_fixed20(self.const_1_120, self.cube_fixed20(x))

        first_numerator = self.add_fixed20(first, self.add_fixed20(second, self.add_fixed20(third, fourth)))
        numerator = self.mul_fixed20(x, first_numerator)
        denominator = self.add_fixed20(self.mul_fixed20(self.const_1_10, self.square_fixed20(x)), self.const_1)

        approx_div = self.newton_division(numerator, denominator, iterations)
        return self.mul_fixed20(self.const_1_2, approx_div)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        results = []
        for x_i in x.flatten():
            x_fixed = self.float_to_fixed20(x_i.item())
            result_fixed = self.custom_silu(x_fixed, iterations=2)
            result_float = self.fixed20_to_float(result_fixed)
            results.append(result_float)
        return torch.tensor(results, device=x.device).view_as(x)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def true_silu(x):
    return x / (1 + np.exp(-x))

def float_to_fixed19(value: float) -> str:
    SCALE = 2**16  # 소수부 16비트
    MAX_INT = 2**18 - 1  # 19비트 최대값
    MIN_INT = -2**18     # 19비트 최소값

    # 스케일링: 실수를 고정소수점 정수로 변환
    scaled = int(round(value * SCALE))

    # 범위 체크
    if not (MIN_INT <= scaled <= MAX_INT):
        raise ValueError(f"Value {value} out of range for 19-bit fixed-point format.")

    # 양수면 그대로, 음수면 2의 보수로
    if scaled >= 0:
        bin_str = format(scaled, '019b')
    else:
        scaled_twos_complement = (1 << 19) + scaled
        bin_str = format(scaled_twos_complement, '019b')

    return bin_str

def fixed19_to_float(bitstring: str) -> float:
    if len(bitstring) != 19:
        raise ValueError("Bitstring must be exactly 19 bits long.")

    is_negative = bitstring[0] == '1'
    int_value = int(bitstring, 2)

    if is_negative:
        # 2의 보수 복원
        int_value -= (1 << 19)

    # 고정소수점 실수 변환
    float_value = int_value / (2**16)
    return float_value

def add_fixed19(a: str, b: str) -> str:
    a_int = int(a, 2)
    b_int = int(b, 2)

    if a[0] == '1':
        a_int -= 1 << 19
    if b[0] == '1':
        b_int -= 1 << 19

    result = a_int + b_int
    if result < 0:
        result = (1 << 19) + result

    return format(result & ((1 << 19) - 1), '019b')


def sub_fixed19(a: str, b: str) -> str:
    a_int = int(a, 2)
    b_int = int(b, 2)

    if a[0] == '1':
        a_int -= 1 << 19
    if b[0] == '1':
        b_int -= 1 << 19

    result = a_int - b_int
    if result < 0:
        result = (1 << 19) + result

    return format(result & ((1 << 19) - 1), '019b')


def mul_fixed19(a: str, b: str) -> str:
    a_int = int(a, 2)
    b_int = int(b, 2)

    if a[0] == '1':
        a_int -= 1 << 19
    if b[0] == '1':
        b_int -= 1 << 19

    # 곱셈 후 스케일 보정 (소수부 16비트니까 오른쪽으로 16비트 시프트)
    result = (a_int * b_int) >> 16
    if result < 0:
        result = (1 << 19) + result

    return format(result & ((1 << 19) - 1), '019b')


def square_fixed19(a: str) -> str:
    a_int = int(a, 2)

    if a[0] == '1':
        a_int -= 1 << 19

    result = (a_int * a_int) >> 16
    if result < 0:
        result = (1 << 19) + result

    return format(result & ((1 << 19) - 1), '019b')

def cube_fixed19(a: str) -> str:
    a_int = int(a, 2)
    if a[0] == '1':
        a_int -= 1 << 19

    # 세제곱 후 스케일 보정 (16비트가 세 번 곱해지므로 16*2 = 32 비트 시프트)
    result = (a_int * a_int * a_int) >> 32
    if result < 0:
        result = (1 << 19) + result

    return format(result & ((1 << 19) - 1), '019b')

def custom_silu(input, iterations=5):
    x = float_to_fixed19(input)
    first = const_1
    second = mul_fixed19(const_1_2, x)
    third = mul_fixed19(const_1_10, square_fixed19(x))
    fourth = mul_fixed19(const_1_120, cube_fixed19(x))

    first_numerator = add_fixed19(first, add_fixed19(second, add_fixed19(third, fourth))) # 1+ x/2 + X^2/10 + x^3/120
    numerator = mul_fixed19(x, first_numerator) # x * (1+ x/2 + X^2/10 + x^3/120)
    denominator = add_fixed19(mul_fixed19(const_1_10, square_fixed19(x)), const_1)

    def get_initial_guess(denom_val):
      d = denom_val
      if const_1 <= d < const_1_020:
          return const_0_999
      elif const_1_020 <= d < const_1_040:
          return const_0_980
      elif const_1_040 <= d < const_1_060:
          return const_0_961
      elif const_1_060 <= d < const_1_080:
          return const_0_943
      else:
          return const_0_925

    def newton_division(num, denom, iterations):
      x_n = get_initial_guess(denom)

      for i in range(iterations):
          #x_n = multiply_32bit(x_n, sub_32bit(CONST_2, multiply_32bit(denom, x_n)))
          x_n = mul_fixed19(x_n, sub_fixed19(const_2, mul_fixed19(denom, x_n)))

      return mul_fixed19(num, x_n)

    approx_div = newton_division(numerator, denominator, iterations)
    result = mul_fixed19(const_1_2, approx_div)  # 마지막에 1/2 곱해져서 원래대로 복원됨

    return result




const_1      = float_to_fixed19(1.0)       # 1
const_1_2    = float_to_fixed19(1.0 / 2.0) # 1/2
const_1_10   = float_to_fixed19(1.0 / 10)  # 1/10
const_1_120  = float_to_fixed19(1.0 / 120) # 1/120
const_2      = float_to_fixed19(2.0)       # 2

const_1_020 = float_to_fixed19(1.020)
const_1_040 = float_to_fixed19(1.040)
const_1_060 = float_to_fixed19(1.060)
const_1_080 = float_to_fixed19(1.080)

const_0_999 = float_to_fixed19(0.999)
const_0_980 = float_to_fixed19(0.980)
const_0_961 = float_to_fixed19(0.961)
const_0_943 = float_to_fixed19(0.943)
const_0_925 = float_to_fixed19(0.925)

####################################################################################################################################
x_vals = np.linspace(-1, 1, 1000)
true_vals = true_silu(x_vals)

results = {}
errors = {}
percent_errors = {}  # 오차율 저장
rmse_vals = {}

for it in range(1, 7):
    custom_vals = np.array([fixed19_to_float(custom_silu(x, iterations=it)) for x in x_vals])
    abs_err = np.abs(custom_vals - true_vals)  # 절대 오차

    percent_err = np.abs((custom_vals - true_vals) / (true_vals + 1e-10)) * 100
    rmse = np.sqrt(np.mean((custom_vals - true_vals) ** 2))

    results[it] = custom_vals
    errors[it] = abs_err
    percent_errors[it] = percent_err
    rmse_vals[it] = rmse

    print(f"Iteration {it}: "
          f"mean abs error = {np.mean(abs_err):.6e}, "
          f"max abs error = {np.max(abs_err):.6e}, "
          f"mean % error = {np.mean(percent_err):.6e}, "
          f"max % error = {np.max(percent_err):.6e}, "
          f"RMSE = {rmse:.6e}")
# 그래프 1: SiLU 비교
plt.figure(figsize=(10, 6))
plt.plot(x_vals, true_vals, label="True SiLU", linewidth=2)
for it in range(1, 7):
    plt.plot(x_vals, results[it], label=f"Iteration {it}", linestyle="--")
plt.title("Custom SiLU vs True SiLU (Different Iterations)")
plt.xlabel("x")
plt.ylabel("SiLU(x)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# 그래프 2: 절대 오차 비교
plt.figure(figsize=(10, 6))
for it in range(1, 7):
    plt.plot(x_vals, errors[it], label=f"Iter {it}")
plt.title("Absolute Error vs Iteration Count")
plt.xlabel("x")
plt.ylabel("Absolute Error")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# 그래프 3: 오차율 비교
plt.figure(figsize=(10, 6))
for it in range(1, 7):
    plt.plot(x_vals, percent_errors[it], label=f"Iter {it}")
plt.title("Percentage Error vs Iteration Count")
plt.xlabel("x")
plt.ylabel("Percentage Error (%)")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def true_silu(x):
    return x / (1 + np.exp(-x))

def float_to_fixed26(value: float) -> str:
    SCALE = 2**23
    MAX_INT = 2**25 - 1
    MIN_INT = -2**25

    scaled = int(round(value * SCALE))

    if not (MIN_INT <= scaled <= MAX_INT):
        raise ValueError(f"Value {value} out of range for 26-bit fixed-point format.")

    if scaled >= 0:
        return format(scaled, '026b')
    else:
        return format((1 << 26) + scaled, '026b')

def fixed26_to_float(bitstring: str) -> float:
    if len(bitstring) != 26:
        raise ValueError("Bitstring must be exactly 26 bits long.")

    is_negative = bitstring[0] == '1'
    int_value = int(bitstring, 2)
    if is_negative:
        int_value -= (1 << 26)
    return int_value / (2**23)

def add_fixed26(a: str, b: str) -> str:
    a_int = int(a, 2)
    b_int = int(b, 2)
    if a[0] == '1':
        a_int -= 1 << 26
    if b[0] == '1':
        b_int -= 1 << 26
    result = a_int + b_int
    if result < 0:
        result = (1 << 26) + result
    return format(result & ((1 << 26) - 1), '026b')

def sub_fixed26(a: str, b: str) -> str:
    a_int = int(a, 2)
    b_int = int(b, 2)
    if a[0] == '1':
        a_int -= 1 << 26
    if b[0] == '1':
        b_int -= 1 << 26
    result = a_int - b_int
    if result < 0:
        result = (1 << 26) + result
    return format(result & ((1 << 26) - 1), '026b')

def mul_fixed26(a: str, b: str) -> str:
    a_int = int(a, 2)
    b_int = int(b, 2)
    if a[0] == '1':
        a_int -= 1 << 26
    if b[0] == '1':
        b_int -= 1 << 26
    result = (a_int * b_int) >> 23
    if result < 0:
        result = (1 << 26) + result
    return format(result & ((1 << 26) - 1), '026b')

def square_fixed26(a: str) -> str:
    a_int = int(a, 2)
    if a[0] == '1':
        a_int -= 1 << 26
    result = (a_int * a_int) >> 23
    if result < 0:
        result = (1 << 26) + result
    return format(result & ((1 << 26) - 1), '026b')

def cube_fixed26(a: str) -> str:
    a_int = int(a, 2)
    if a[0] == '1':
        a_int -= 1 << 26
    result = (a_int * a_int * a_int) >> 46
    if result < 0:
        result = (1 << 26) + result
    return format(result & ((1 << 26) - 1), '026b')

# Constants
const_1      = float_to_fixed26(1.0)
const_1_2    = float_to_fixed26(1.0 / 2.0)
const_1_10   = float_to_fixed26(1.0 / 10)
const_1_120  = float_to_fixed26(1.0 / 120)
const_2      = float_to_fixed26(2.0)

const_1_020 = float_to_fixed26(1.020)
const_1_040 = float_to_fixed26(1.040)
const_1_060 = float_to_fixed26(1.060)
const_1_080 = float_to_fixed26(1.080)

const_0_999 = float_to_fixed26(0.999)
const_0_980 = float_to_fixed26(0.980)
const_0_961 = float_to_fixed26(0.961)
const_0_943 = float_to_fixed26(0.943)
const_0_925 = float_to_fixed26(0.925)

def custom_silu_fixed26(input, iterations=5):
    x = float_to_fixed26(input)
    first = const_1
    second = mul_fixed26(const_1_2, x)
    third = mul_fixed26(const_1_10, square_fixed26(x))
    fourth = mul_fixed26(const_1_120, cube_fixed26(x))

    first_numerator = add_fixed26(first, add_fixed26(second, add_fixed26(third, fourth)))
    numerator = mul_fixed26(x, first_numerator)
    denominator = add_fixed26(mul_fixed26(const_1_10, square_fixed26(x)), const_1)

    def get_initial_guess(denom_val):
        d = denom_val
        if const_1 <= d < const_1_020:
            return const_0_999
        elif const_1_020 <= d < const_1_040:
            return const_0_980
        elif const_1_040 <= d < const_1_060:
            return const_0_961
        elif const_1_060 <= d < const_1_080:
            return const_0_943
        else:
            return const_0_925

    def newton_division(num, denom, iterations):
        x_n = get_initial_guess(denom)
        for i in range(iterations):
            x_n = mul_fixed26(x_n, sub_fixed26(const_2, mul_fixed26(denom, x_n)))
        return mul_fixed26(num, x_n)

    approx_div = newton_division(numerator, denominator, iterations)
    result = mul_fixed26(const_1_2, approx_div)
    return result

# 테스트 및 시각화
x_vals = np.linspace(-1, 1, 1000)
true_vals = true_silu(x_vals)

results = {}
errors = {}
percent_errors = {}
rmse_vals = {}

for it in range(1, 7):
    custom_vals = np.array([fixed26_to_float(custom_silu_fixed26(x, iterations=it)) for x in x_vals])
    abs_err = np.abs(custom_vals - true_vals)
    percent_err = np.abs((custom_vals - true_vals) / (true_vals + 1e-10)) * 100
    rmse = np.sqrt(np.mean((custom_vals - true_vals) ** 2))

    results[it] = custom_vals
    errors[it] = abs_err
    percent_errors[it] = percent_err
    rmse_vals[it] = rmse

    print(f"Iteration {it}: "
          f"mean abs error = {np.mean(abs_err):.6e}, "
          f"max abs error = {np.max(abs_err):.6e}, "
          f"mean % error = {np.mean(percent_err):.6e}, "
          f"max % error = {np.max(percent_err):.6e}, "
          f"RMSE = {rmse:.6e}")

# 그래프
plt.figure(figsize=(10, 6))
plt.plot(x_vals, true_vals, label="True SiLU", linewidth=2)
for it in range(1, 7):
    plt.plot(x_vals, results[it], label=f"Iteration {it}", linestyle="--")
plt.title("Custom SiLU (26bit) vs True SiLU")
plt.xlabel("x")
plt.ylabel("SiLU(x)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
for it in range(1, 7):
    plt.plot(x_vals, errors[it], label=f"Iter {it}")
plt.title("Absolute Error (26bit)")
plt.xlabel("x")
plt.ylabel("Absolute Error")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
for it in range(1, 7):
    plt.plot(x_vals, percent_errors[it], label=f"Iter {it}")
plt.title("Percentage Error (26bit)")
plt.xlabel("x")
plt.ylabel("Percentage Error (%)")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### 값 확인

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class KANLinear(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            # scalar multiplier per (out, in)
            self.spline_scaler = nn.Parameter(torch.Tensor(out_features, in_features))

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (torch.rand(self.grid_size + 1, self.in_features, self.out_features) - 0.5)
                * self.scale_noise / self.grid_size
            )
            coeff = self.curve2coeff(
                self.grid.T[self.spline_order:-self.spline_order], noise
            )
            self.spline_weight.data.copy_(coeff)
            if self.enable_standalone_scale_spline:
                nn.init.ones_(self.spline_scaler)

    def b_splines(self, x: torch.Tensor):
        B, D = x.shape
        K = self.grid_size + self.spline_order
        x_unsq = x.unsqueeze(-1)
        bases = ((x_unsq >= self.grid[:, :-1]) & (x_unsq < self.grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            left = (x_unsq - self.grid[:, :-(k+1)]) / (self.grid[:, k:-1] - self.grid[:, :-(k+1)]) * bases[:, :, :-1]
            right = (self.grid[:, k+1:] - x_unsq) / (self.grid[:, k+1:] - self.grid[:, 1:-k]) * bases[:, :, 1:]
            bases = left + right
        return bases.contiguous()  # (B, D, K)

    @property
    def scaled_spline_weight(self):
        if self.enable_standalone_scale_spline:
            return self.spline_weight * self.spline_scaler.unsqueeze(-1)
        return self.spline_weight

    def forward(self, x: torch.Tensor):
        B = x.shape[0]
        x_flat = x.view(B, self.in_features)
        base_out = F.linear(self.base_activation(x_flat), self.base_weight)
        bs = self.b_splines(x_flat)
        spline_out = F.linear(
            bs.view(B, -1), self.scaled_spline_weight.view(self.out_features, -1)
        )
        return base_out + spline_out

    def forward_debug_one(
        self,
        x: torch.Tensor,
        debug_in_idx: int,
        debug_out_idx: int,
        debug_batch_idx: int = None,
    ):
        """
        디버그 정보:
          - xi: x_i 값
          - Bi: B-spline basis vector (length K)
          - Ci: spline coefficient vector C_i (length K)
          - w_s_scalar: global spline multiplier scalar
          - w_b: base weight scalar w_b
          - y_j: 최종 출력 scalar or (B,)
        """
        B = x.shape[0]
        x_flat = x.view(B, self.in_features)
        out = self.forward(x_flat)

        # input scalar
        xi = x_flat[:, debug_in_idx]
        # basis Bi(x)
        Bs = self.b_splines(x_flat)
        Bi = Bs[:, debug_in_idx, :]  # (B, K)
        # coefficient Ci for that basis
        Ci = self.spline_weight[debug_out_idx, debug_in_idx, :].detach()  # (K,)
        # global multiplier scalar
        if self.enable_standalone_scale_spline:
            w_s_scalar = self.spline_scaler[debug_out_idx, debug_in_idx].detach()
        else:
            w_s_scalar = torch.tensor(self.scale_spline, device=x.device)
        # base weight
        w_b = self.base_weight[debug_out_idx, debug_in_idx].detach()
        # output
        y = out[:, debug_out_idx]

        if debug_batch_idx is not None:
            return (
                xi[debug_batch_idx],      # scalar
                Bi[debug_batch_idx, :],   # (K,)
                Ci,                       # (K,)
                w_s_scalar,               # scalar
                w_b,                      # scalar
                y[debug_batch_idx],       # scalar
            )
        return xi, Bi, Ci, w_s_scalar, w_b, y

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        A = self.b_splines(x).transpose(0, 1)
        Bm = y.transpose(0, 1)
        sol = torch.linalg.lstsq(A, Bm).solution
        return sol.permute(2, 0, 1).contiguous()

class KAN(nn.Module):
    def __init__(self, layers_hidden, **kwargs):
        super().__init__()
        self.layers = nn.ModuleList(
            KANLinear(in_f, out_f, **kwargs)
            for in_f, out_f in zip(layers_hidden, layers_hidden[1:])
        )

    def forward(
        self,
        x: torch.Tensor,
        debug_layer: int = None,
        debug_in_idx: int = None,
        debug_out_idx: int = None,
        debug_batch_idx: int = None,
    ):
        debug = None
        for idx, layer in enumerate(self.layers):
            if idx == debug_layer:
                debug = layer.forward_debug_one(
                    x, debug_in_idx, debug_out_idx, debug_batch_idx
                )
                x = layer(x)
            else:
                x = layer(x)
        return (x, debug) if debug is not None else x


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

kan = KAN([784,64,10], grid_size=5, spline_order=3).to(device)
x   = torch.randn(32,784, device=device)

y, debug = kan(
    x,
    debug_layer=0,
    debug_in_idx=0,
    debug_out_idx=10,
    debug_batch_idx=5
)

# unpack
xi, Bi, Ci, w_s_scalar, w_b, y_j = debug

print("xi:", xi)
print("Bi (basis):", Bi)
print("Ci (coeffs):", Ci)
print("w_s (scalar multiplier):", w_s_scalar)
print("w_b:", w_b)
print("y_j:", y_j)


## latency

In [ ]:
import torch
import torch.nn.functional as F
import math


class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        x = x.reshape(-1, self.in_features)

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        output = base_output + spline_output

        output = output.reshape(*original_shape[:-1], self.out_features)
        return output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin=0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype=torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )


class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KAN, self).__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

In [ ]:
start = torch.cuda.Event(enable_timing=True)
end   = torch.cuda.Event(enable_timing=True)

# 워밍업
for _ in range(10):
    _ = model(x)
torch.cuda.synchronize()

# 측정
start.record()
_ = model(x)
end.record()
torch.cuda.synchronize()  # 커널 종료 대기

print(f"CUDA kernel elapsed: {start.elapsed_time(end):.3f} ms")


In [ ]:
# 1) trace 한 번
traced = torch.jit.trace(model, x)

# 2) (워밍업)
for _ in range(10):
    _ = traced(x)

# 3) 측정
start = time.perf_counter()
_ = traced(x)
end = time.perf_counter()
print(f"{(end-start)*1e6:.1f} µs")


In [ ]:
import time

# 워밍업…
for _ in range(10):
    _ = traced(x)
if device.type == "cuda":
    torch.cuda.synchronize()

# 측정
t0 = time.perf_counter()
_  = traced(x)
if device.type == "cuda":
    torch.cuda.synchronize()
t1 = time.perf_counter()

print(f"{(t1-t0)*1e6:.1f} µs")


In [ ]:
import torch

# 1) device 정의
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) 모델·입력 생성
kan = KAN([1, 1], grid_size=5, spline_order=3).to(device)
model = KAN([1, 1], grid_size=5, spline_order=3).to(device)
x   = torch.randn(1, 1, device=device)

# 3) (원하시면 연산 테스트)
with torch.no_grad():
    y = kan(x)
    print("Output:", y)


In [ ]:
traced = torch.jit.trace(kan, x)

# 4) 워밍업
with torch.no_grad():
    for _ in range(10):
        _ = traced(x)
    if device.type == "cuda":
        torch.cuda.synchronize()


# 5) 순수 퓨전 커널 실행 시간 측정
start = torch.cuda.Event(enable_timing=True)
end   = torch.cuda.Event(enable_timing=True)

start.record()
_ = traced(x)
end.record()
torch.cuda.synchronize()

print(f"Pure JIT kernel time: {start.elapsed_time(end)*1e3:.0f} µs")

In [ ]:
import time
import torch

# 1) 환경 세팅
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
kan    = KAN([1, 1], grid_size=5, spline_order=3).to(device)
kan.eval()

# 2) 테스트할 레이어·노드 인덱스
layer_idx  = 0   # 첫 번째 레이어
in_idx     = 0   # 입력 채널 0
out_idx    = 0   # 출력 채널 0

layer = kan.layers[layer_idx]

# 3) 샘플 입력 (batch=1, feature=1)
x = torch.randn(1, 1, device=device)

# 4) 한 노드 연산 함수 정의
def compute_node(layer, x_scalar, in_idx, out_idx):
    # 1) silu
    a = layer.base_activation(x_scalar)  # shape=(1,)
    base = layer.base_weight[out_idx, in_idx] * a  # shape=(1,)

    # 2) B-spline basis (shape=(1, K)) → take [0, in_idx, :]
    Bs = layer.b_splines(x)[0, in_idx]                # shape=(K,)
    Ci = layer.spline_weight[out_idx, in_idx]         # shape=(K,)
    spline_term = (Ci * Bs).sum()                     # scalar

    # 3) standalone 스칼라 곱
    if layer.enable_standalone_scale_spline:
        spline_term = layer.spline_scaler[out_idx, in_idx] * spline_term

    # 4) 최종 합산
    return (base + spline_term).item()

# 5) 워밍업 (캐시·동기화 비용 제거)
with torch.no_grad():
    for _ in range(10):
        _ = compute_node(layer, x, in_idx, out_idx)
    if device.type == "cuda":
        torch.cuda.synchronize()

# 6) 반복 측정
iters = 1000
t0 = time.perf_counter()
for _ in range(iters):
    _ = compute_node(layer, x, in_idx, out_idx)
if device.type == "cuda":
    torch.cuda.synchronize()
t1 = time.perf_counter()

avg_us = (t1 - t0) / iters * 1e6
print(f"한 노드 순수 KAN 연산 평균: {avg_us:.2f} µs")


In [ ]:
import time
import torch
import torch.nn.functional as F
import math
import statistics


class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        # -------------------------------------------------------------------
        # 1) grid (control points) 생성
        # -------------------------------------------------------------------
        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        # grid.shape = (in_features, grid_size + 2 * spline_order + 1)
        # ex) in_features=1, grid_size=5, spline_order=3 → 1×(5+6+1)=1×12
        self.register_buffer("grid", grid)

        # -------------------------------------------------------------------
        # 2) trainable 파라미터: base_weight, spline_weight, (옵션) spline_scaler
        # -------------------------------------------------------------------
        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            # spline_scaler를 쓰지 않으면 spline_weight 에 직접 scale_spline 곱해서 초기화함
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        # -------------------------------------------------------------------
        # 3) 그 외 설정
        # -------------------------------------------------------------------
        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()  # 예: torch.nn.SiLU()
        self.grid_eps = grid_eps

        # 파라미터 초기화
        self.reset_parameters()

    def reset_parameters(self):
        # -------------------------------------------------------------------
        # ① base_weight: kaiming_uniform
        # -------------------------------------------------------------------
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)

        # -------------------------------------------------------------------
        # ② spline_weight 초기화:
        #    noise를 랜덤하게 생성한 뒤, curve2coeff를 통해 “계수”로 바꿔서 저장
        # -------------------------------------------------------------------
        with torch.no_grad():
            # noise.shape = (grid_size+1, in_features, out_features)
            noise = (
                (torch.rand(self.grid_size + 1, self.in_features, self.out_features) - 0.5)
                * self.scale_noise
                / self.grid_size
            )
            # curve2coeff: 입력 x 값(=grid control points), 출력 noise를 받아서
            #    B-spline 계수로 바꿔 줌
            # grid.T[self.spline_order : -self.spline_order] 의 shape이 (in_features, grid_size+1)
            # noise.shape = (grid_size+1, in_features, out_features)
            coeff = self.curve2coeff(
                self.grid.T[self.spline_order : -self.spline_order],  # (in_features, grid_size+1)
                noise,  # (grid_size+1, in_features, out_features)
            )
            # coeff.shape = (out_features, in_features, grid_size + spline_order)
            # scale_spline이 standalone이면, spline_scaler를 kaiming_uniform으로 init
            if not self.enable_standalone_scale_spline:
                self.spline_weight.data.copy_(coeff * self.scale_spline)
            else:
                self.spline_weight.data.copy_(coeff)  # scale은 spline_scaler 가 담당
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        x: (batch, in_features)
        return: (batch, in_features, grid_size + spline_order)
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = self.grid
        # x.unsqueeze(-1): (batch, in_features, 1)
        x_exp = x.unsqueeze(-1)

        # ① 먼저 “원 조각(piecewise) 0차 B-spline”을 구함 → bases 초기화
        #    bases.shape = (batch, in_features, grid_size + 2*spline_order)
        bases = ((x_exp >= grid[:, :-1]) & (x_exp < grid[:, 1:])).to(x.dtype)

        # ② 차수(order)만큼 De Boor 재귀식으로 업그레이드
        for k in range(1, self.spline_order + 1):
            left_num   = (x_exp - grid[:, : -(k + 1)])  # shape (batch, in, len−(k+1))
            left_den   = (grid[:, k:-1] - grid[:, : -(k + 1)])
            left_frac  = left_num / left_den
            left_term  = left_frac * bases[:, :, :-1]

            right_num  = (grid[:, k + 1 :] - x_exp)
            right_den  = (grid[:, k + 1 :] - grid[:, 1:(-k)])
            right_frac = right_num / right_den
            right_term = right_frac * bases[:, :, 1:]

            bases = left_term + right_term

        # 최종 크기: (batch, in_features, grid_size + spline_order)
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        # spline_weight: (out_features, in_features, grid_size+spline_order)
        if not self.enable_standalone_scale_spline:
            return self.spline_weight * self.scale_spline
        else:
            # spline_scaler: (out_features, in_features)
            # unsqueeze(-1) → (out_features, in_features, 1)
            return self.spline_weight * self.spline_scaler.unsqueeze(-1)

    def forward(self, x: torch.Tensor):
        """
        x.shape = (batch, in_features)
        return: (batch, out_features)
        """
        assert x.size(-1) == self.in_features
        original_shape = x.shape
        B, D = x.shape

        x_flat = x.reshape(-1, self.in_features)  # (B, D)

        # 〈연산 부문〉: SiLU, Linear(base), B-spline, Linear(spline), 합산
        # → 외부에서 “forward() 전체” 시간을 측정하는 것으로 바꿔야 합니다.
        activated = self.base_activation(x_flat)  # (B, D)
        base_out = F.linear(activated, self.base_weight)  # (B, out_features)

        bs = self.b_splines(x_flat).view(B, -1)  # (B, D * (grid_size + spline_order))
        spline_out = F.linear(
            bs,  # (B, D * K)
            self.scaled_spline_weight.view(self.out_features, -1),  # (out_features, D * K)
        )  # (B, out_features)

        out = base_out + spline_out  # (B, out_features)
        return out.reshape(*original_shape[:-1], self.out_features)


class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KAN, self).__init__()
        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid=False):
        for layer in self.layers:
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )


# ==========================================================
# 아래는 “forward() 로 직접 재는 대신” 외부에서 반복 측정하여
#   ① CPU 상에서 순수 연산(SiLU + B-spline + 합산) 시간 측정
#   ② GPU 상에서 순수 퓨전 커널(SiLU+Linear+Spline 모두 퓨전된) 실행 시간 측정
# ==========================================================
if __name__ == "__main__":
    # ----------------------------------------------------------
    # 1) CPU 환경에서 측정: time.perf_counter() + 반복/통계
    # ----------------------------------------------------------
    device = torch.device("cpu")
    kan_cpu = KAN([1, 1], grid_size=5, spline_order=3).to(device)
    kan_cpu.eval()
    x_cpu = torch.randn(1, 1, device=device)

    # 충분히 워밍업 (페이지 폴트, 함수 캐시 등 제거)
    with torch.no_grad():
        for _ in range(200):
            _ = kan_cpu(x_cpu)

    times_us = []
    reps = 2000
    with torch.no_grad():
        for _ in range(reps):
            t0 = time.perf_counter()
            _ = kan_cpu(x_cpu)  # (SiLU + B-spline + 합산) 연산 전체
            t1 = time.perf_counter()
            times_us.append((t1 - t0) * 1e6)  # µs 단위

    mean_us   = statistics.mean(times_us)
    median_us = statistics.median(times_us)
    print(f"--- CPU 측정 ---")
    print(f"평균 forward 시간: {mean_us:.2f} µs")
    print(f"중앙값 forward 시간: {median_us:.2f} µs\n")

    # ----------------------------------------------------------
    # 2) GPU 환경에서 측정: torch.cuda.Event + JIT 퓨전 커널 실행
    # ----------------------------------------------------------
    if torch.cuda.is_available():
        device = torch.device("cuda")
        kan_gpu = KAN([1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1], grid_size=5, spline_order=3).to(device)
        kan_gpu.eval()
        x_gpu = torch.randn(1, 1, device=device)

        # 2-1) TorchScript 트레이스 (연산 퓨전)
        traced_gpu = torch.jit.trace(kan_gpu, x_gpu)

        # 충분히 워밍업: GPU 캐시, 퓨전 커널 컴파일 등
        with torch.no_grad():
            for _ in range(100):
                _ = traced_gpu(x_gpu)
            torch.cuda.synchronize()

        # 순수 퓨전 커널 측정
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)

        times_ms = []
        reps_gpu = 500
        with torch.no_grad():
            for _ in range(reps_gpu):
                start.record()
                _ = traced_gpu(x_gpu)
                end.record()
                torch.cuda.synchronize()
                times_ms.append(start.elapsed_time(end))

        mean_ms   = sum(times_ms) / len(times_ms)
        median_ms = sorted(times_ms)[len(times_ms) // 2]
        print(f"--- GPU 측정 ---")
        print(f"평균 퓨전 커널 시간: {mean_ms*1e3:.1f} µs")
        print(f"중앙값 퓨전 커널 시간: {median_ms*1e3:.1f} µs")
    else:
        print("CUDA 사용 불가: GPU 측정 생략")
